In [ ]:

import pandas as pd
from collections import Counter

FILE_PATH = r"D:\PowerBI_Project\ecommerce.csv"
CHUNKSIZE = 500_000

unique_users = set()
unique_sessions = set()

event_user_counts = Counter()
event_session_counts = Counter()

purchase_price_stats = {
    "count": 0,
    "sum": 0.0,
    "min": float("inf"),
    "max": float("-inf"),
    "negative": 0,
    "zero": 0
}

global_user_ids = set()
global_session_ids = set()

total_duplicates = 0
total_rows = 0

for chunk in pd.read_csv(
    FILE_PATH,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    total_rows += len(chunk)

    # Unique users and sessions
    unique_users.update(chunk["user_id"].dropna().unique())
    unique_sessions.update(chunk["user_session"].dropna().unique())

    # Event-level user/session counts
    for event, group in chunk.groupby("event_type"):
        event_user_counts[event] += group["user_id"].nunique()
        event_session_counts[event] += group["user_session"].nunique()

    # Purchase price validation
    purchases = chunk[chunk["event_type"] == "purchase"]
    prices = purchases["price"].dropna()

    if len(prices) > 0:
        purchase_price_stats["count"] += len(prices)
        purchase_price_stats["sum"] += prices.sum()
        purchase_price_stats["min"] = min(
            purchase_price_stats["min"], prices.min()
        )
        purchase_price_stats["max"] = max(
            purchase_price_stats["max"], prices.max()
        )
        purchase_price_stats["negative"] += (prices < 0).sum()
        purchase_price_stats["zero"] += (prices == 0).sum()

    # Exact duplicates within each chunk
    total_duplicates += chunk.duplicated().sum()

print("========== USER & SESSION ==========")
print("Total rows:", total_rows)
print("Unique users:", len(unique_users))
print("Unique sessions:", len(unique_sessions))

print("\n========== PURCHASE PRICE ==========")
print(purchase_price_stats)

print("\n========== EVENT USER COUNTS ==========")
print(event_user_counts)

print("\n========== EVENT SESSION COUNTS ==========")
print(event_session_counts)

print("\n========== DUPLICATES ==========")
print("Within-chunk duplicate rows:", total_duplicates)
print("Note: Cross-chunk duplicates are not yet checked.")


In [ ]:

import pandas as pd
from pathlib import Path
from collections import Counter

FILE_PATH = Path(r"D:\PowerBI_Project\ecommerce.csv")
CHUNKSIZE = 500_000

# Fresh variables: no previous accumulated results
total_rows = 0
event_counts = Counter()
unique_users = set()
unique_sessions = set()

purchase_count = 0
purchase_price_sum = 0.0
purchase_price_min = float("inf")
purchase_price_max = float("-inf")

duplicate_within_chunks = 0
chunk_count = 0

print("File path:", FILE_PATH.resolve())
print("File size (MB):", round(FILE_PATH.stat().st_size / 1024**2, 2))

for chunk in pd.read_csv(
    FILE_PATH,
    chunksize=CHUNKSIZE,
    low_memory=False
):
    chunk_count += 1
    total_rows += len(chunk)

    event_counts.update(
        chunk["event_type"].value_counts(dropna=False).to_dict()
    )

    unique_users.update(chunk["user_id"].dropna().unique())
    unique_sessions.update(chunk["user_session"].dropna().unique())

    purchases = chunk.loc[
        chunk["event_type"].eq("purchase"),
        "price"
    ].dropna()

    purchase_count += len(purchases)

    if len(purchases):
        purchase_price_sum += purchases.sum()
        purchase_price_min = min(
            purchase_price_min, purchases.min()
        )
        purchase_price_max = max(
            purchase_price_max, purchases.max()
        )

    duplicate_within_chunks += chunk.duplicated().sum()

print("\n========== CLEAN AUDIT ==========")
print("Chunks processed:", chunk_count)
print("Total rows:", total_rows)
print("Event counts:", event_counts)
print("Unique users:", len(unique_users))
print("Unique sessions:", len(unique_sessions))

print("\n========== PURCHASE VALIDATION ==========")
print("Purchase rows:", purchase_count)
print("Purchase price sum:", round(purchase_price_sum, 2))
print("Purchase price min:", purchase_price_min)
print("Purchase price max:", purchase_price_max)

print("\n========== DUPLICATE CHECK ==========")
print("Within-chunk duplicates:", duplicate_within_chunks)


In [ ]:

from pathlib import Path
import os
import pandas as pd
import hashlib

FILE_PATH = Path(r"D:\PowerBI_Project\ecommerce.csv")

print("========== FILE IDENTITY ==========")
print("Absolute Path:", FILE_PATH.resolve())
print("File Exists:", FILE_PATH.exists())
print("File Size (Bytes):", FILE_PATH.stat().st_size)
print("File Size (MB):", round(FILE_PATH.stat().st_size / 1024**2, 2))
print("Last Modified:", pd.Timestamp(
    FILE_PATH.stat().st_mtime, unit="s"
))

print("\n========== FILE HEADER ==========")
with open(FILE_PATH, "rb") as f:
    print(f.read(500))

print("\n========== SAMPLE ==========")
sample = pd.read_csv(FILE_PATH, nrows=10)
display(sample)

print("\n========== SAMPLE HASH ==========")
with open(FILE_PATH, "rb") as f:
    first_mb = f.read(1024 * 1024)

print("First 1MB SHA256:", hashlib.sha256(first_mb).hexdigest())


In [ ]:
%pip install duckdb

In [ ]:

import duckdb
import os

FILE_PATH = r"D:\PowerBI_Project\ecommerce.csv"
TEMP_DIR = r"D:\PowerBI_Project\duckdb_temp"

os.makedirs(TEMP_DIR, exist_ok=True)

con = duckdb.connect()

con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 4")
con.execute(f"SET temp_directory = '{TEMP_DIR}'")

query = """
WITH grouped AS (
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session,
        COUNT(*) AS row_count
    FROM read_csv_auto(
        ?,
        header = true,
        sample_size = 100000
    )
    GROUP BY
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session
)
SELECT
    SUM(row_count) AS total_rows,

    SUM(
        CASE WHEN row_count > 1
        THEN row_count - 1 ELSE 0 END
    ) AS exact_duplicate_rows,

    SUM(
        CASE WHEN row_count > 1
        THEN 1 ELSE 0 END
    ) AS duplicate_groups,

    MAX(row_count) AS maximum_group_size

FROM grouped
"""

result = con.execute(query, [FILE_PATH]).df()

print("========== CORRECTED GLOBAL DUPLICATE AUDIT ==========")
display(result)

total = int(result.loc[0, "total_rows"])
duplicates = int(result.loc[0, "exact_duplicate_rows"])

print("\nTotal Rows:", total)
print("Exact Duplicate Rows:", duplicates)
print(
    "Duplicate Percentage:",
    round(duplicates / total * 100, 6),
    "%"
)


In [ ]:

import duckdb
import pandas as pd

FILE_PATH = r"D:\PowerBI_Project\ecommerce.csv"

con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 4")
con.execute(
    r"SET temp_directory = 'D:\PowerBI_Project\duckdb_temp'"
)

# All nine fields define an exact duplicate
grouped_cte = """
WITH grouped AS (
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session,
        COUNT(*) AS row_count
    FROM read_csv_auto(?, header=true, sample_size=100000)
    GROUP BY
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session
)
"""

# 1. Duplicate distribution by event type
query_event = grouped_cte + """
SELECT
    event_type,
    SUM(row_count - 1) AS duplicate_rows,
    COUNT(*) AS duplicate_groups,
    MAX(row_count) AS maximum_group_size
FROM grouped
WHERE row_count > 1
GROUP BY event_type
ORDER BY duplicate_rows DESC
"""

event_result = con.execute(
    query_event, [FILE_PATH]
).df()

print("========== DUPLICATES BY EVENT TYPE ==========")
display(event_result)

# 2. Duplicate distribution by date
query_date = grouped_cte + """
SELECT
    SUBSTR(event_time, 1, 10) AS event_date,
    SUM(row_count - 1) AS duplicate_rows,
    COUNT(*) AS duplicate_groups,
    MAX(row_count) AS maximum_group_size
FROM grouped
WHERE row_count > 1
GROUP BY event_date
ORDER BY duplicate_rows DESC
"""

date_result = con.execute(
    query_date, [FILE_PATH]
).df()

print("========== DUPLICATES BY DATE ==========")
display(date_result)

# 3. Top 20 most repeated exact records
query_top = grouped_cte + """
SELECT *
FROM grouped
WHERE row_count > 1
ORDER BY row_count DESC
LIMIT 20
"""

top_result = con.execute(
    query_top, [FILE_PATH]
).df()

print("========== TOP 20 DUPLICATE GROUPS ==========")
display(top_result)


In [ ]:

# Phase 1.3.2 — Duplicate Distribution by Date

query_date = grouped_cte + """
SELECT
    CAST(event_time AS DATE) AS event_date,
    SUM(row_count - 1) AS duplicate_rows,
    COUNT(*) AS duplicate_groups,
    MAX(row_count) AS maximum_group_size
FROM grouped
WHERE row_count > 1
GROUP BY CAST(event_time AS DATE)
ORDER BY duplicate_rows DESC
"""

date_result = con.execute(
    query_date, [FILE_PATH]
).df()

print("========== DUPLICATES BY DATE ==========")
display(date_result)


In [ ]:

# Phase 1.3.3 — Top 20 Duplicate Groups

query_top = grouped_cte + """
SELECT
    event_time,
    event_type,
    product_id,
    category_id,
    category_code,
    brand,
    price,
    user_id,
    user_session,
    row_count,
    row_count - 1 AS duplicate_extra_rows
FROM grouped
WHERE row_count > 1
ORDER BY row_count DESC, event_time
LIMIT 20
"""

top_result = con.execute(
    query_top, [FILE_PATH]
).df()

print("========== TOP 20 DUPLICATE GROUPS ==========")
display(top_result)


In [ ]:

# Analyze how many times each exact record appears

query_multiplicity = grouped_cte + """
SELECT
    event_type,
    row_count AS occurrences,
    COUNT(*) AS duplicate_groups,
    SUM(row_count - 1) AS extra_duplicate_rows
FROM grouped
WHERE row_count > 1
GROUP BY event_type, row_count
ORDER BY event_type, occurrences
"""

multiplicity_result = con.execute(
    query_multiplicity, [FILE_PATH]
).df()

print("========== DUPLICATE MULTIPLICITY ==========")
display(multiplicity_result)

# Summary by event type and multiplicity range
query_buckets = grouped_cte + """
SELECT
    event_type,
    CASE
        WHEN row_count = 2 THEN '2 occurrences'
        WHEN row_count BETWEEN 3 AND 5 THEN '3-5 occurrences'
        WHEN row_count BETWEEN 6 AND 10 THEN '6-10 occurrences'
        WHEN row_count BETWEEN 11 AND 20 THEN '11-20 occurrences'
        ELSE 'More than 20 occurrences'
    END AS occurrence_range,
    COUNT(*) AS duplicate_groups,
    SUM(row_count - 1) AS extra_duplicate_rows,
    MAX(row_count) AS maximum_group_size
FROM grouped
WHERE row_count > 1
GROUP BY event_type, occurrence_range
ORDER BY event_type, occurrence_range
"""

bucket_result = con.execute(
    query_buckets, [FILE_PATH]
).df()

print("\n========== DUPLICATE GROUP SIZE SUMMARY ==========")
display(bucket_result)


In [ ]:

# Phase 1.3.5 — Targeted Duplicate Investigation

query_investigation = grouped_cte + """
SELECT
    event_time,
    event_type,
    product_id,
    category_id,
    category_code,
    brand,
    price,
    user_id,
    user_session,
    row_count,
    row_count - 1 AS extra_duplicate_rows
FROM grouped
WHERE
    (event_type = 'purchase' AND row_count > 1)
    OR
    (event_type = 'cart' AND row_count > 20)
ORDER BY
    event_type,
    row_count DESC,
    event_time
"""

investigation_result = con.execute(
    query_investigation, [FILE_PATH]
).df()

print("========== PURCHASE DUPLICATES + HIGH-FREQUENCY CART ==========")
print("Total duplicate groups:", len(investigation_result))
display(investigation_result)

# Separate summary
print("\n========== INVESTIGATION SUMMARY ==========")

display(
    investigation_result.groupby("event_type").agg(
        duplicate_groups=("row_count", "size"),
        extra_duplicate_rows=("extra_duplicate_rows", "sum"),
        max_occurrences=("row_count", "max")
    ).reset_index()
)


In [ ]:

# Get all events from sessions containing:
# 1. duplicated Purchase groups
# 2. Cart groups occurring more than 20 times

query_session_context = """
WITH target_sessions AS (
    SELECT DISTINCT
        user_id,
        user_session
    FROM (
        SELECT
            user_id,
            user_session,
            event_type,
            event_time,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            COUNT(*) AS n
        FROM read_csv_auto(
            ?,
            header=true,
            sample_size=100000
        )
        GROUP BY
            user_id,
            user_session,
            event_type,
            event_time,
            product_id,
            category_id,
            category_code,
            brand,
            price
    )
    WHERE
        (event_type = 'purchase' AND n > 1)
        OR
        (event_type = 'cart' AND n > 20)
)
SELECT
    e.user_id,
    e.user_session,
    e.event_time,
    e.event_type,
    e.product_id,
    e.price
FROM read_csv_auto(
    ?,
    header=true,
    sample_size=100000
) e
INNER JOIN target_sessions t
    ON e.user_id = t.user_id
    AND e.user_session = t.user_session
ORDER BY
    e.user_id,
    e.user_session,
    e.event_time
LIMIT 500
"""

context_result = con.execute(
    query_session_context,
    [FILE_PATH, FILE_PATH]
).df()

print("========== TARGET SESSION CONTEXT (MAX 500 ROWS) ==========")
display(context_result)


In [ ]:

# Target Session Integrity Audit
# No changes to the original CSV

query_target_summary = """
WITH grouped AS (
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session,
        COUNT(*) AS row_count
    FROM read_csv_auto(
        ?,
        header = true,
        sample_size = 100000
    )
    GROUP BY
        event_time, event_type, product_id,
        category_id, category_code, brand,
        price, user_id, user_session
),

target_sessions AS (
    SELECT
        user_id,
        user_session,
        MAX(CASE WHEN event_type = 'purchase'
                 AND row_count > 1 THEN 1 ELSE 0 END)
            AS has_duplicate_purchase,
        MAX(CASE WHEN event_type = 'cart'
                 AND row_count > 20 THEN 1 ELSE 0 END)
            AS has_high_frequency_cart
    FROM grouped
    GROUP BY user_id, user_session
    HAVING
        MAX(CASE WHEN event_type = 'purchase'
                 AND row_count > 1 THEN 1 ELSE 0 END) = 1
        OR
        MAX(CASE WHEN event_type = 'cart'
                 AND row_count > 20 THEN 1 ELSE 0 END) = 1
),

session_summary AS (
    SELECT
        e.user_id,
        e.user_session,
        t.has_duplicate_purchase,
        t.has_high_frequency_cart,
        COUNT(*) AS total_events,
        COUNT_IF(e.event_type = 'view') AS view_events,
        COUNT_IF(e.event_type = 'cart') AS cart_events,
        COUNT_IF(e.event_type = 'purchase') AS purchase_events,
        COUNT(DISTINCT e.event_time) AS distinct_event_times,
        COUNT(DISTINCT e.product_id) AS distinct_products,
        MIN(e.event_time) AS first_event,
        MAX(e.event_time) AS last_event
    FROM read_csv_auto(
        ?,
        header = true,
        sample_size = 100000
    ) e
    INNER JOIN target_sessions t
        ON e.user_id = t.user_id
        AND e.user_session = t.user_session
    GROUP BY
        e.user_id,
        e.user_session,
        t.has_duplicate_purchase,
        t.has_high_frequency_cart
)

SELECT *
FROM session_summary
ORDER BY
    has_duplicate_purchase DESC,
    has_high_frequency_cart DESC,
    total_events DESC
"""

target_summary = con.execute(
    query_target_summary,
    [FILE_PATH, FILE_PATH]
).df()

print("========== TARGET SESSION SUMMARY ==========")
print("Target sessions:", len(target_summary))
display(target_summary)

print("\n========== OVERALL EVENT SUMMARY ==========")
display(
    target_summary[
        ["view_events", "cart_events", "purchase_events", "total_events"]
    ].sum().to_frame("event_count")
)


In [ ]:

query_purchase_impact = grouped_cte + """
SELECT
    COUNT(*) AS duplicate_purchase_groups,
    SUM(row_count - 1) AS extra_purchase_rows,
    SUM(price * (row_count - 1)) AS extra_purchase_gmv,
    SUM(price) AS duplicated_group_price_sum,
    MAX(row_count) AS maximum_group_size
FROM grouped
WHERE event_type = 'purchase'
  AND row_count > 1
"""

purchase_impact = con.execute(
    query_purchase_impact,
    [FILE_PATH]
).df()

print("========== DUPLICATE PURCHASE GMV IMPACT ==========")
display(purchase_impact)


In [ ]:

query_flag_summary = """
WITH grouped AS (
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session,
        COUNT(*) AS row_count
    FROM read_csv_auto(
        ?,
        header=true,
        sample_size=100000
    )
    GROUP BY
        event_time, event_type, product_id,
        category_id, category_code, brand,
        price, user_id, user_session
),

target_sessions AS (
    SELECT
        user_id,
        user_session,
        MAX(CASE
            WHEN event_type = 'purchase' AND row_count > 1
            THEN 1 ELSE 0
        END) AS has_duplicate_purchase,
        MAX(CASE
            WHEN event_type = 'cart' AND row_count > 20
            THEN 1 ELSE 0
        END) AS has_high_frequency_cart
    FROM grouped
    GROUP BY user_id, user_session
    HAVING
        MAX(CASE
            WHEN event_type = 'purchase' AND row_count > 1
            THEN 1 ELSE 0
        END) = 1
        OR
        MAX(CASE
            WHEN event_type = 'cart' AND row_count > 20
            THEN 1 ELSE 0
        END) = 1
)

SELECT
    has_duplicate_purchase,
    has_high_frequency_cart,
    COUNT(*) AS session_count
FROM target_sessions
GROUP BY
    has_duplicate_purchase,
    has_high_frequency_cart
ORDER BY
    has_duplicate_purchase DESC,
    has_high_frequency_cart DESC
"""

flag_summary = con.execute(
    query_flag_summary,
    [FILE_PATH]
).df()

print("========== TARGET SESSION CLASSIFICATION ==========")
display(flag_summary)


In [ ]:

import duckdb
import os

RAW_FILE = r"D:\PowerBI_Project\ecommerce.csv"

CLEAN_FILE = r"D:\PowerBI_Project\ecommerce_deduplicated.parquet"

TEMP_DIR = r"D:\PowerBI_Project\duckdb_temp"

os.makedirs(TEMP_DIR, exist_ok=True)

con = duckdb.connect()

con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 4")
con.execute(
    "SET temp_directory = 'D:/PowerBI_Project/duckdb_temp'"
)

# Never overwrite the original CSV.
# Create a separate analytical copy.
if os.path.exists(CLEAN_FILE):
    raise FileExistsError(
        "Clean file already exists. Do not overwrite it. "
        "Please inspect or rename the existing file first."
    )

print("Starting deduplication...")
print("Original:", RAW_FILE)
print("Output:", CLEAN_FILE)

con.execute("""
COPY (
    SELECT DISTINCT *
    FROM read_csv_auto(
        ?,
        header = true,
        sample_size = 100000
    )
)
TO ?
(FORMAT PARQUET, COMPRESSION ZSTD)
""", [RAW_FILE, CLEAN_FILE])

print("\nDeduplication completed.")
print("Clean file exists:", os.path.exists(CLEAN_FILE))
print(
    "Clean file size (MB):",
    round(os.path.getsize(CLEAN_FILE) / 1024**2, 2)
)


In [ ]:

CLEAN_FILE = r"D:\PowerBI_Project\ecommerce_deduplicated.parquet"

validation = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT user_session) AS unique_sessions,

    COUNT(*) FILTER (
        WHERE event_type = 'view'
    ) AS view_events,

    COUNT(*) FILTER (
        WHERE event_type = 'cart'
    ) AS cart_events,

    COUNT(*) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_events,

    SUM(price) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_event_gmv

FROM read_parquet(?)
""", [CLEAN_FILE]).df()

print("========== CLEAN DATA VALIDATION ==========")
display(validation)


In [ ]:

CLEAN_FILE = r"D:\PowerBI_Project\ecommerce_deduplicated.parquet"

verification = con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) - COUNT(DISTINCT (
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session
    )) AS duplicate_rows,

    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT user_session) AS unique_sessions,

    SUM(price) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_event_gmv

FROM read_parquet(?)
""", [CLEAN_FILE]).df()

print("========== FINAL CLEAN DATA VERIFICATION ==========")
display(verification)


In [ ]:

import duckdb

CLEAN_FILE = r"D:\PowerBI_Project\ecommerce_deduplicated.parquet"

con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 4")
con.execute(
    "SET temp_directory = 'D:/PowerBI_Project/duckdb_temp'"
)

query_user_profile = """
WITH user_profile AS (
    SELECT
        user_id,

        COUNT(*) AS total_events,

        COUNT(*) FILTER (
            WHERE event_type = 'view'
        ) AS view_events,

        COUNT(*) FILTER (
            WHERE event_type = 'cart'
        ) AS cart_events,

        COUNT(*) FILTER (
            WHERE event_type = 'purchase'
        ) AS purchase_events,

        SUM(price) FILTER (
            WHERE event_type = 'purchase'
        ) AS total_spent,

        COUNT(DISTINCT user_session) AS total_sessions,

        MIN(event_time) AS first_activity,
        MAX(event_time) AS last_activity

    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
    GROUP BY user_id
)

SELECT *
FROM user_profile
"""

user_profile = con.execute(
    query_user_profile,
    [CLEAN_FILE]
).df()

print("========== USER PROFILE ==========")
print("Total users:", len(user_profile))
display(user_profile.head(10))

print("\n========== USER EVENT SUMMARY ==========")
display(
    user_profile[
        ["total_events", "view_events", "cart_events",
         "purchase_events", "total_sessions", "total_spent"]
    ].describe().round(2)
)

print("\n========== PURCHASE USER DISTRIBUTION ==========")
display(
    user_profile.assign(
        purchase_status=user_profile["purchase_events"].apply(
            lambda x: (
                "No Purchase" if x == 0
                else "Single Purchase" if x == 1
                else "Repeat Purchase"
            )
        )
    )["purchase_status"].value_counts().to_frame("users")
)


In [ ]:

query_distribution = """
WITH user_profile AS (
    SELECT
        user_id,
        COUNT(*) AS total_events,

        COUNT(*) FILTER (
            WHERE event_type = 'view'
        ) AS view_events,

        COUNT(*) FILTER (
            WHERE event_type = 'cart'
        ) AS cart_events,

        COUNT(*) FILTER (
            WHERE event_type = 'purchase'
        ) AS purchase_events,

        SUM(price) FILTER (
            WHERE event_type = 'purchase'
        ) AS total_spent,

        COUNT(DISTINCT user_session) AS total_sessions

    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
    GROUP BY user_id
)

SELECT
    QUANTILE_CONT(total_events, 0.50) AS events_p50,
    QUANTILE_CONT(total_events, 0.75) AS events_p75,
    QUANTILE_CONT(total_events, 0.90) AS events_p90,
    QUANTILE_CONT(total_events, 0.95) AS events_p95,
    QUANTILE_CONT(total_events, 0.99) AS events_p99,

    QUANTILE_CONT(purchase_events, 0.50) AS purchase_p50,
    QUANTILE_CONT(purchase_events, 0.90) AS purchase_p90,
    QUANTILE_CONT(purchase_events, 0.99) AS purchase_p99,

    QUANTILE_CONT(total_sessions, 0.50) AS sessions_p50,
    QUANTILE_CONT(total_sessions, 0.90) AS sessions_p90,
    QUANTILE_CONT(total_sessions, 0.99) AS sessions_p99,

    QUANTILE_CONT(total_spent, 0.50) AS spend_p50,
    QUANTILE_CONT(total_spent, 0.75) AS spend_p75,
    QUANTILE_CONT(total_spent, 0.90) AS spend_p90,
    QUANTILE_CONT(total_spent, 0.95) AS spend_p95,
    QUANTILE_CONT(total_spent, 0.99) AS spend_p99

FROM user_profile
"""

distribution_result = con.execute(
    query_distribution,
    [CLEAN_FILE]
).df()

print("========== USER DISTRIBUTION QUANTILES ==========")
display(distribution_result.T)


In [ ]:

query_top_users = """
SELECT
    user_id,
    COUNT(*) AS total_events,

    COUNT(*) FILTER (
        WHERE event_type = 'view'
    ) AS view_events,

    COUNT(*) FILTER (
        WHERE event_type = 'cart'
    ) AS cart_events,

    COUNT(*) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_events,

    SUM(price) FILTER (
        WHERE event_type = 'purchase'
    ) AS total_spent,

    COUNT(DISTINCT user_session) AS total_sessions

FROM read_parquet(?)
WHERE user_id IS NOT NULL
GROUP BY user_id
HAVING COUNT(*) FILTER (
    WHERE event_type = 'purchase'
) > 0
ORDER BY total_spent DESC
LIMIT 20
"""

top_users = con.execute(
    query_top_users,
    [CLEAN_FILE]
).df()

print("========== TOP 20 USERS BY SPEND ==========")
display(top_users)


In [ ]:

CLEAN_FILE = r"D:\PowerBI_Project\ecommerce_deduplicated.parquet"

base_profile = """
SELECT
    user_id,
    COUNT(*) AS total_events,

    COUNT(*) FILTER (
        WHERE event_type = 'view'
    ) AS view_events,

    COUNT(*) FILTER (
        WHERE event_type = 'cart'
    ) AS cart_events,

    COUNT(*) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_events,

    SUM(price) FILTER (
        WHERE event_type = 'purchase'
    ) AS total_spent,

    COUNT(DISTINCT user_session) AS total_sessions

FROM read_parquet(?)
WHERE user_id IS NOT NULL
GROUP BY user_id
"""

# Top 20 by purchase frequency
top_purchase_users = con.execute(
    f"""
    WITH profile AS ({base_profile})
    SELECT *
    FROM profile
    ORDER BY purchase_events DESC
    LIMIT 20
    """,
    [CLEAN_FILE]
).df()

print("========== TOP 20 BY PURCHASE EVENTS ==========")
display(top_purchase_users)

# Top 20 by number of sessions
top_session_users = con.execute(
    f"""
    WITH profile AS ({base_profile})
    SELECT *
    FROM profile
    ORDER BY total_sessions DESC
    LIMIT 20
    """,
    [CLEAN_FILE]
).df()

print("\n========== TOP 20 BY TOTAL SESSIONS ==========")
display(top_session_users)

# Top 20 by spending
top_spend_users = con.execute(
    f"""
    WITH profile AS ({base_profile})
    SELECT *
    FROM profile
    WHERE total_spent IS NOT NULL
    ORDER BY total_spent DESC
    LIMIT 20
    """,
    [CLEAN_FILE]
).df()

print("\n========== TOP 20 BY TOTAL SPEND ==========")
display(top_spend_users)


In [ ]:

query_extreme_dates = """
WITH user_profile AS (
    SELECT
        user_id,
        COUNT(*) FILTER (
            WHERE event_type = 'purchase'
        ) AS purchase_events,
        COUNT(DISTINCT user_session) AS total_sessions,
        SUM(price) FILTER (
            WHERE event_type = 'purchase'
        ) AS total_spent,
        MIN(event_time) AS first_activity,
        MAX(event_time) AS last_activity,
        DATE_DIFF(
            'day',
            MIN(event_time),
            MAX(event_time)
        ) AS active_span_days
    FROM read_parquet(?)
    GROUP BY user_id
)

SELECT *
FROM user_profile
WHERE purchase_events >= 100
   OR total_sessions >= 100
   OR total_spent >= 10000
ORDER BY total_spent DESC NULLS LAST
LIMIT 100
"""

extreme_users = con.execute(
    query_extreme_dates,
    [CLEAN_FILE]
).df()

print("========== EXTREME USER DATE SPAN ==========")
display(extreme_users)


In [ ]:

target_users = [
    512475445,  # 7,400 sessions
    526731152,  # 2,773 sessions
    514649263,  # 2,336 sessions
    512386086   # 321 purchase events
]

query_session_audit = """
WITH session_profile AS (
    SELECT
        user_id,
        user_session,
        COUNT(*) AS events_per_session,

        COUNT(*) FILTER (
            WHERE event_type = 'view'
        ) AS views,

        COUNT(*) FILTER (
            WHERE event_type = 'cart'
        ) AS carts,

        COUNT(*) FILTER (
            WHERE event_type = 'purchase'
        ) AS purchases,

        MIN(event_time) AS session_start,
        MAX(event_time) AS session_end

    FROM read_parquet(?)
    WHERE user_id IN (
        512475445,
        526731152,
        514649263,
        512386086
    )
    GROUP BY user_id, user_session
),

user_summary AS (
    SELECT
        user_id,
        COUNT(*) AS session_count,

        COUNT(*) FILTER (
            WHERE events_per_session = 1
        ) AS single_event_sessions,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE events_per_session = 1
            ) / COUNT(*),
            2
        ) AS single_event_session_pct,

        ROUND(AVG(events_per_session), 2)
            AS avg_events_per_session,

        MEDIAN(events_per_session)
            AS median_events_per_session,

        MAX(events_per_session)
            AS max_events_per_session,

        SUM(views) AS total_views,
        SUM(carts) AS total_carts,
        SUM(purchases) AS total_purchases,

        MIN(session_start) AS first_activity,
        MAX(session_end) AS last_activity

    FROM session_profile
    GROUP BY user_id
)

SELECT *
FROM user_summary
ORDER BY session_count DESC
"""

session_audit = con.execute(
    query_session_audit,
    [CLEAN_FILE]
).df()

print("========== EXTREME USER SESSION AUDIT ==========")
display(session_audit)


In [ ]:

query_session_distribution = """
WITH session_profile AS (
    SELECT
        user_id,
        user_session,
        COUNT(*) AS events_per_session
    FROM read_parquet(?)
    WHERE user_id IN (
        512475445,
        526731152,
        514649263,
        512386086
    )
    GROUP BY user_id, user_session
)

SELECT
    user_id,

    CASE
        WHEN events_per_session = 1 THEN '1 event'
        WHEN events_per_session = 2 THEN '2 events'
        WHEN events_per_session BETWEEN 3 AND 5 THEN '3-5 events'
        WHEN events_per_session BETWEEN 6 AND 10 THEN '6-10 events'
        ELSE 'More than 10 events'
    END AS session_size,

    COUNT(*) AS session_count

FROM session_profile
GROUP BY user_id, session_size
ORDER BY user_id, session_size
"""

session_distribution = con.execute(
    query_session_distribution,
    [CLEAN_FILE]
).df()

print("========== SESSION SIZE DISTRIBUTION ==========")
display(session_distribution)


In [ ]:

query_global_session = """
WITH session_profile AS (
    SELECT
        user_id,
        user_session,

        COUNT(*) AS total_events,

        COUNT(*) FILTER (
            WHERE event_type = 'view'
        ) AS view_events,

        COUNT(*) FILTER (
            WHERE event_type = 'cart'
        ) AS cart_events,

        COUNT(*) FILTER (
            WHERE event_type = 'purchase'
        ) AS purchase_events

    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
      AND user_session IS NOT NULL

    GROUP BY user_id, user_session
)

SELECT
    COUNT(*) AS total_sessions,

    COUNT(*) FILTER (
        WHERE total_events = 1
    ) AS single_event_sessions,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE total_events = 1
        ) / COUNT(*), 2
    ) AS single_event_session_pct,

    COUNT(*) FILTER (
        WHERE total_events >= 2
    ) AS multi_event_sessions,

    COUNT(*) FILTER (
        WHERE cart_events > 0
    ) AS sessions_with_cart,

    COUNT(*) FILTER (
        WHERE purchase_events > 0
    ) AS sessions_with_purchase,

    COUNT(*) FILTER (
        WHERE view_events > 0
          AND cart_events = 0
          AND purchase_events = 0
    ) AS view_only_sessions,

    ROUND(AVG(total_events), 2)
        AS avg_events_per_session,

    MEDIAN(total_events)
        AS median_events_per_session,

    MAX(total_events)
        AS max_events_per_session

FROM session_profile
"""

global_session_summary = con.execute(
    query_global_session,
    [CLEAN_FILE]
).df()

print("========== GLOBAL SESSION SUMMARY ==========")
display(global_session_summary)


In [ ]:

query_global_session_distribution = """
WITH session_profile AS (
    SELECT
        user_id,
        user_session,
        COUNT(*) AS total_events
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
      AND user_session IS NOT NULL
    GROUP BY user_id, user_session
)

SELECT
    CASE
        WHEN total_events = 1 THEN '1 event'
        WHEN total_events = 2 THEN '2 events'
        WHEN total_events BETWEEN 3 AND 5 THEN '3-5 events'
        WHEN total_events BETWEEN 6 AND 10 THEN '6-10 events'
        WHEN total_events BETWEEN 11 AND 20 THEN '11-20 events'
        ELSE 'More than 20 events'
    END AS session_size,

    COUNT(*) AS session_count,

    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS session_percentage

FROM session_profile

GROUP BY session_size

ORDER BY
    CASE session_size
        WHEN '1 event' THEN 1
        WHEN '2 events' THEN 2
        WHEN '3-5 events' THEN 3
        WHEN '6-10 events' THEN 4
        WHEN '11-20 events' THEN 5
        ELSE 6
    END
"""

global_session_distribution = con.execute(
    query_global_session_distribution,
    [CLEAN_FILE]
).df()

print("========== GLOBAL SESSION SIZE DISTRIBUTION ==========")
display(global_session_distribution)


In [ ]:

query_session_identity = """
WITH session_mapping AS (
    SELECT
        user_session,
        COUNT(DISTINCT user_id) AS user_count,
        COUNT(*) AS event_count
    FROM read_parquet(?)
    WHERE user_session IS NOT NULL
    GROUP BY user_session
)

SELECT
    COUNT(*) AS unique_session_ids,

    SUM(user_count) AS session_user_pairs,

    COUNT(*) FILTER (
        WHERE user_count > 1
    ) AS session_ids_with_multiple_users,

    SUM(user_count - 1) FILTER (
        WHERE user_count > 1
    ) AS additional_user_session_pairs,

    MAX(user_count) AS max_users_per_session,

    SUM(event_count) AS mapped_events

FROM session_mapping
"""

identity_result = con.execute(
    query_session_identity,
    [CLEAN_FILE]
).df()

print("========== SESSION IDENTITY RECONCILIATION ==========")
display(identity_result)


In [ ]:

query_session_nulls = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE user_id IS NULL
    ) AS null_user_id_rows,

    COUNT(*) FILTER (
        WHERE user_session IS NULL
    ) AS null_session_rows,

    COUNT(*) FILTER (
        WHERE user_id IS NULL
          AND user_session IS NULL
    ) AS both_null_rows

FROM read_parquet(?)
"""

null_result = con.execute(
    query_session_nulls,
    [CLEAN_FILE]
).df()

print("========== SESSION KEY NULL AUDIT ==========")
display(null_result)


In [ ]:

query_collision_distribution = """
WITH mapping AS (
    SELECT
        user_session,
        COUNT(DISTINCT user_id) AS user_count,
        COUNT(*) AS event_count
    FROM read_parquet(?)
    WHERE user_session IS NOT NULL
    GROUP BY user_session
)

SELECT
    user_count,
    COUNT(*) AS session_id_count,
    SUM(event_count) AS event_rows
FROM mapping
WHERE user_count > 1
GROUP BY user_count
ORDER BY user_count
"""

collision_distribution = con.execute(
    query_collision_distribution,
    [CLEAN_FILE]
).df()

print("========== CROSS-USER SESSION DISTRIBUTION ==========")
display(collision_distribution)


In [ ]:

query_collision_details = """
WITH collision_ids AS (
    SELECT user_session
    FROM read_parquet(?)
    WHERE user_session IS NOT NULL
    GROUP BY user_session
    HAVING COUNT(DISTINCT user_id) > 1
)

SELECT
    e.user_session,
    e.user_id,
    COUNT(*) AS event_count,

    COUNT(*) FILTER (
        WHERE e.event_type = 'view'
    ) AS view_events,

    COUNT(*) FILTER (
        WHERE e.event_type = 'cart'
    ) AS cart_events,

    COUNT(*) FILTER (
        WHERE e.event_type = 'purchase'
    ) AS purchase_events,

    MIN(e.event_time) AS first_event,
    MAX(e.event_time) AS last_event

FROM read_parquet(?) e
INNER JOIN collision_ids c
    ON e.user_session = c.user_session

GROUP BY e.user_session, e.user_id
ORDER BY e.user_session, e.user_id
"""

collision_details = con.execute(
    query_collision_details,
    [CLEAN_FILE, CLEAN_FILE]
).df()

print("========== CROSS-USER SESSION DETAILS ==========")
print("Rows:", len(collision_details))
display(collision_details.head(50))


In [ ]:

query_null_session_rows = """
SELECT *
FROM read_parquet(?)
WHERE user_session IS NULL
"""

null_session_records = con.execute(
    query_null_session_rows,
    [CLEAN_FILE]
).df()

print("========== NULL SESSION RECORDS ==========")
display(null_session_records)


In [ ]:

query_collision_overlap = """
WITH user_session_profile AS (
    SELECT
        user_session,
        user_id,
        COUNT(*) AS event_count,
        COUNT(*) FILTER (WHERE event_type = 'view') AS views,
        COUNT(*) FILTER (WHERE event_type = 'cart') AS carts,
        COUNT(*) FILTER (WHERE event_type = 'purchase') AS purchases,
        MIN(event_time) AS first_event,
        MAX(event_time) AS last_event
    FROM read_parquet(?)
    WHERE user_session IS NOT NULL
    GROUP BY user_session, user_id
),
collisions AS (
    SELECT *
    FROM user_session_profile
    WHERE user_session IN (
        SELECT user_session
        FROM user_session_profile
        GROUP BY user_session
        HAVING COUNT(*) > 1
    )
),
pairs AS (
    SELECT
        a.user_session,
        a.user_id AS user_a,
        b.user_id AS user_b,
        a.event_count AS events_a,
        b.event_count AS events_b,
        a.views AS views_a,
        b.views AS views_b,
        a.carts AS carts_a,
        b.carts AS carts_b,
        a.purchases AS purchases_a,
        b.purchases AS purchases_b,
        a.first_event AS first_a,
        a.last_event AS last_a,
        b.first_event AS first_b,
        b.last_event AS last_b,

        CASE
            WHEN a.first_event <= b.last_event
             AND b.first_event <= a.last_event
            THEN 1 ELSE 0
        END AS time_interval_overlap

    FROM collisions a
    JOIN collisions b
      ON a.user_session = b.user_session
     AND a.user_id < b.user_id
)

SELECT
    COUNT(*) AS user_pairs,
    SUM(time_interval_overlap) AS overlapping_pairs,
    COUNT(*) - SUM(time_interval_overlap) AS non_overlapping_pairs,
    SUM(events_a + events_b) AS total_events_in_pairs
FROM pairs
"""

overlap_summary = con.execute(
    query_collision_overlap,
    [CLEAN_FILE]
).df()

print("========== COLLISION TIME OVERLAP SUMMARY ==========")
display(overlap_summary)


In [ ]:

query_collision_behaviour = """
WITH collision_ids AS (
    SELECT user_session
    FROM read_parquet(?)
    WHERE user_session IS NOT NULL
    GROUP BY user_session
    HAVING COUNT(DISTINCT user_id) > 1
)

SELECT
    e.event_type,
    COUNT(*) AS event_rows,
    COUNT(DISTINCT e.user_id) AS affected_users,
    COUNT(DISTINCT e.user_session) AS affected_sessions
FROM read_parquet(?) e
JOIN collision_ids c
    ON e.user_session = c.user_session
GROUP BY e.event_type
ORDER BY event_rows DESC
"""

collision_behaviour = con.execute(
    query_collision_behaviour,
    [CLEAN_FILE, CLEAN_FILE]
).df()

print("========== COLLISION EVENT TYPE DISTRIBUTION ==========")
display(collision_behaviour)


In [ ]:

query_overlap_details = """
WITH profile AS (
    SELECT
        user_session,
        user_id,
        COUNT(*) AS event_count,
        COUNT(*) FILTER (WHERE event_type = 'view') AS views,
        COUNT(*) FILTER (WHERE event_type = 'cart') AS carts,
        COUNT(*) FILTER (WHERE event_type = 'purchase') AS purchases,
        MIN(event_time) AS first_event,
        MAX(event_time) AS last_event
    FROM read_parquet(?)
    WHERE user_session IS NOT NULL
    GROUP BY user_session, user_id
),
pairs AS (
    SELECT
        a.user_session,
        a.user_id AS user_a,
        b.user_id AS user_b,
        a.event_count AS events_a,
        b.event_count AS events_b,
        a.views AS views_a,
        b.views AS views_b,
        a.carts AS carts_a,
        b.carts AS carts_b,
        a.purchases AS purchases_a,
        b.purchases AS purchases_b,
        a.first_event AS first_a,
        a.last_event AS last_a,
        b.first_event AS first_b,
        b.last_event AS last_b
    FROM profile a
    JOIN profile b
      ON a.user_session = b.user_session
     AND a.user_id < b.user_id
    WHERE a.first_event <= b.last_event
      AND b.first_event <= a.last_event
)
SELECT *
FROM pairs
ORDER BY user_session
"""

overlap_details = con.execute(
    query_overlap_details,
    [CLEAN_FILE]
).df()

print("========== OVERLAPPING USER PAIRS ==========")
display(overlap_details)


In [ ]:

query_collision_purchases = """
WITH collision_ids AS (
    SELECT user_session
    FROM read_parquet(?)
    WHERE user_session IS NOT NULL
    GROUP BY user_session
    HAVING COUNT(DISTINCT user_id) > 1
)
SELECT
    e.user_session,
    e.user_id,
    e.event_time,
    e.event_type,
    e.product_id,
    e.category_code,
    e.brand,
    e.price
FROM read_parquet(?) e
JOIN collision_ids c
    ON e.user_session = c.user_session
WHERE e.event_type = 'purchase'
ORDER BY e.event_time
"""

collision_purchases = con.execute(
    query_collision_purchases,
    [CLEAN_FILE, CLEAN_FILE]
).df()

print("========== PURCHASE ATTRIBUTION ==========")
display(collision_purchases)


In [ ]:

OUTPUT_SESSION_FILE = (
    r"D:\PowerBI_Project\ecommerce_session_ready.parquet"
)

query_build_session = """
COPY (
    SELECT
        *,
        CASE
            WHEN user_session IS NOT NULL
            THEN CAST(user_id AS VARCHAR)
                 || '_'
                 || user_session
            ELSE NULL
        END AS Session_Key
    FROM read_parquet(?)
)
TO ?
(FORMAT PARQUET, COMPRESSION ZSTD)
"""

con.execute(
    query_build_session,
    [CLEAN_FILE, OUTPUT_SESSION_FILE]
)

print("Session-ready file created:")
print(OUTPUT_SESSION_FILE)


In [ ]:

query_validate_session = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT user_id) AS unique_users,

    COUNT(DISTINCT user_session) AS global_session_ids,

    COUNT(DISTINCT Session_Key) AS unique_session_keys,

    COUNT(*) FILTER (
        WHERE user_session IS NULL
    ) AS null_session_rows,

    COUNT(*) FILTER (
        WHERE Session_Key IS NULL
    ) AS null_session_key_rows

FROM read_parquet(?)
"""

session_validation = con.execute(
    query_validate_session,
    [OUTPUT_SESSION_FILE]
).df()

print("========== SESSION KEY VALIDATION ==========")
display(session_validation)


In [ ]:

USER_PROFILE_FILE = (
    r"D:\PowerBI_Project\user_profile_v3.parquet"
)

query_user_profile = """
COPY (
    SELECT
        user_id,

        COUNT(*) AS total_events,

        COUNT(*) FILTER (
            WHERE event_type = 'view'
        ) AS view_events,

        COUNT(*) FILTER (
            WHERE event_type = 'cart'
        ) AS cart_events,

        COUNT(*) FILTER (
            WHERE event_type = 'purchase'
        ) AS purchase_events,

        SUM(
            CASE
                WHEN event_type = 'purchase'
                THEN price
                ELSE 0
            END
        ) AS total_spent,

        COUNT(DISTINCT Session_Key) AS total_sessions,

        MIN(event_time) AS first_activity,

        MAX(event_time) AS last_activity

    FROM read_parquet(?)
    GROUP BY user_id
)
TO ?
(FORMAT PARQUET, COMPRESSION ZSTD)
"""

con.execute(
    query_user_profile,
    [OUTPUT_SESSION_FILE, USER_PROFILE_FILE]
)

print("User profile created:")
print(USER_PROFILE_FILE)


In [ ]:

query_user_profile_validation = """
SELECT
    COUNT(*) AS total_users,

    SUM(total_events) AS total_events,

    SUM(view_events) AS view_events,

    SUM(cart_events) AS cart_events,

    SUM(purchase_events) AS purchase_events,

    SUM(total_spent) AS purchase_event_gmv,

    SUM(total_sessions) AS user_session_pairs,

    COUNT(*) FILTER (
        WHERE purchase_events = 0
    ) AS users_without_purchase,

    COUNT(*) FILTER (
        WHERE purchase_events = 1
    ) AS single_purchase_users,

    COUNT(*) FILTER (
        WHERE purchase_events >= 2
    ) AS repeat_purchase_users

FROM read_parquet(?)
"""

user_profile_validation = con.execute(
    query_user_profile_validation,
    [USER_PROFILE_FILE]
).df()

print("========== USER PROFILE VALIDATION ==========")
display(user_profile_validation)


In [ ]:

query_product_integrity = """
WITH product_attributes AS (
    SELECT
        product_id,

        COUNT(DISTINCT category_id) AS category_id_count,

        COUNT(DISTINCT category_code) AS category_code_count,

        COUNT(DISTINCT brand) AS brand_count,

        COUNT(*) FILTER (
            WHERE category_code IS NULL
        ) AS missing_category_rows,

        COUNT(*) FILTER (
            WHERE brand IS NULL
        ) AS missing_brand_rows,

        COUNT(*) FILTER (
            WHERE event_type = 'purchase'
        ) AS purchase_events

    FROM read_parquet(?)
    GROUP BY product_id
)

SELECT
    COUNT(*) AS total_products,

    COUNT(*) FILTER (
        WHERE category_id_count > 1
    ) AS products_multiple_category_ids,

    COUNT(*) FILTER (
        WHERE category_code_count > 1
    ) AS products_multiple_category_codes,

    COUNT(*) FILTER (
        WHERE brand_count > 1
    ) AS products_multiple_brands,

    COUNT(*) FILTER (
        WHERE category_code_count = 0
    ) AS products_missing_category_code,

    COUNT(*) FILTER (
        WHERE brand_count = 0
    ) AS products_missing_brand,

    SUM(purchase_events) AS total_purchase_events

FROM product_attributes
"""

product_integrity = con.execute(
    query_product_integrity,
    [OUTPUT_SESSION_FILE]
).df()

print("========== PRODUCT INTEGRITY ==========")
display(product_integrity)


In [ ]:

query_product_conflicts = """
SELECT
    product_id,

    COUNT(DISTINCT category_id) AS category_id_count,

    COUNT(DISTINCT category_code) AS category_code_count,

    COUNT(DISTINCT brand) AS brand_count,

    STRING_AGG(
        DISTINCT category_code,
        ' | '
    ) AS category_codes,

    STRING_AGG(
        DISTINCT brand,
        ' | '
    ) AS brands

FROM read_parquet(?)
GROUP BY product_id

HAVING
    COUNT(DISTINCT category_id) > 1
    OR COUNT(DISTINCT category_code) > 1
    OR COUNT(DISTINCT brand) > 1

ORDER BY product_id
"""

product_conflicts = con.execute(
    query_product_conflicts,
    [OUTPUT_SESSION_FILE]
).df()

print("========== PRODUCT ATTRIBUTE CONFLICTS ==========")
print("Conflict products:", len(product_conflicts))
display(product_conflicts.head(50))


In [ ]:

query_brand_conflict_details = """
SELECT
    product_id,
    brand,
    event_type,
    COUNT(*) AS event_rows,
    COUNT(*) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_events,
    SUM(price) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_event_gmv,
    MIN(event_time) AS first_seen,
    MAX(event_time) AS last_seen

FROM read_parquet(?)

WHERE product_id IN (
    SELECT product_id
    FROM read_parquet(?)
    GROUP BY product_id
    HAVING COUNT(DISTINCT brand) > 1
)

GROUP BY product_id, brand, event_type
ORDER BY product_id, brand, event_type
"""

brand_conflict_details = con.execute(
    query_brand_conflict_details,
    [OUTPUT_SESSION_FILE, OUTPUT_SESSION_FILE]
).df()

print("========== BRAND CONFLICT DETAILS ==========")
print("Rows:", len(brand_conflict_details))
display(brand_conflict_details.head(100))


In [ ]:

query_purchase_attribute_coverage = """
SELECT
    COUNT(*) AS purchase_events,

    COUNT(*) FILTER (
        WHERE category_code IS NULL
    ) AS purchases_missing_category,

    COUNT(*) FILTER (
        WHERE brand IS NULL
    ) AS purchases_missing_brand,

    COUNT(DISTINCT product_id) AS unique_purchase_products,

    COUNT(DISTINCT product_id) FILTER (
        WHERE category_code IS NULL
    ) AS purchase_products_missing_category,

    COUNT(DISTINCT product_id) FILTER (
        WHERE brand IS NULL
    ) AS purchase_products_missing_brand,

    SUM(price) FILTER (
        WHERE category_code IS NULL
    ) AS gmv_missing_category,

    SUM(price) FILTER (
        WHERE brand IS NULL
    ) AS gmv_missing_brand

FROM read_parquet(?)
WHERE event_type = 'purchase'
"""

purchase_attribute_coverage = con.execute(
    query_purchase_attribute_coverage,
    [OUTPUT_SESSION_FILE]
).df()

print("========== PURCHASE ATTRIBUTE COVERAGE ==========")
display(purchase_attribute_coverage)


In [ ]:

query_attribute_missing_patterns = """
SELECT
    COUNT(*) AS total_products,

    COUNT(*) FILTER (
        WHERE missing_brand_rows > 0
    ) AS products_with_any_missing_brand,

    COUNT(*) FILTER (
        WHERE missing_category_rows > 0
    ) AS products_with_any_missing_category,

    COUNT(*) FILTER (
        WHERE missing_brand_rows > 0
          AND brand_count > 0
    ) AS products_with_mixed_brand_missingness,

    COUNT(*) FILTER (
        WHERE missing_category_rows > 0
          AND category_code_count > 0
    ) AS products_with_mixed_category_missingness

FROM (
    SELECT
        product_id,

        COUNT(DISTINCT brand) AS brand_count,

        COUNT(DISTINCT category_code) AS category_code_count,

        COUNT(*) FILTER (
            WHERE brand IS NULL
        ) AS missing_brand_rows,

        COUNT(*) FILTER (
            WHERE category_code IS NULL
        ) AS missing_category_rows

    FROM read_parquet(?)
    GROUP BY product_id
) p
"""

missing_patterns = con.execute(
    query_attribute_missing_patterns,
    [OUTPUT_SESSION_FILE]
).df()

print("========== PRODUCT MISSINGNESS PATTERNS ==========")
display(missing_patterns)


In [ ]:

query_purchase_brand_conflicts = """
WITH conflicting_products AS (
    SELECT product_id
    FROM read_parquet(?)
    GROUP BY product_id
    HAVING COUNT(DISTINCT brand) > 1
),
purchase_brand_profile AS (
    SELECT
        e.product_id,
        COUNT(*) AS purchase_events,
        COUNT(DISTINCT e.brand) AS purchase_brand_count,
        COUNT(*) FILTER (
            WHERE e.brand IS NULL
        ) AS missing_brand_purchases,
        SUM(e.price) AS purchase_gmv,
        STRING_AGG(
            DISTINCT e.brand,
            ' | '
        ) AS purchase_brands
    FROM read_parquet(?) e
    INNER JOIN conflicting_products c
        ON e.product_id = c.product_id
    WHERE e.event_type = 'purchase'
    GROUP BY e.product_id
)
SELECT
    COUNT(*) AS conflicting_products_with_purchases,

    COUNT(*) FILTER (
        WHERE purchase_brand_count > 1
    ) AS products_with_multiple_purchase_brands,

    COUNT(*) FILTER (
        WHERE purchase_brand_count = 1
    ) AS products_with_one_purchase_brand,

    COUNT(*) FILTER (
        WHERE purchase_brand_count = 0
    ) AS products_with_only_missing_purchase_brand,

    SUM(purchase_events) AS purchase_events,

    SUM(purchase_gmv) AS purchase_gmv

FROM purchase_brand_profile
"""

purchase_brand_conflict_summary = con.execute(
    query_purchase_brand_conflicts,
    [OUTPUT_SESSION_FILE, OUTPUT_SESSION_FILE]
).df()

print("========== PURCHASE BRAND CONFLICT SUMMARY ==========")
display(purchase_brand_conflict_summary)


In [ ]:

query_multiple_purchase_brands = """
WITH conflicting_products AS (
    SELECT product_id
    FROM read_parquet(?)
    GROUP BY product_id
    HAVING COUNT(DISTINCT brand) > 1
)
SELECT
    product_id,
    brand,
    COUNT(*) AS purchase_events,
    SUM(price) AS purchase_gmv,
    MIN(event_time) AS first_purchase,
    MAX(event_time) AS last_purchase
FROM read_parquet(?)
WHERE event_type = 'purchase'
  AND product_id IN (
      SELECT product_id FROM conflicting_products
  )
GROUP BY product_id, brand
ORDER BY product_id, brand
"""

multiple_purchase_brands = con.execute(
    query_multiple_purchase_brands,
    [OUTPUT_SESSION_FILE, OUTPUT_SESSION_FILE]
).df()

print("========== MULTIPLE PURCHASE BRANDS ==========")
display(multiple_purchase_brands)


In [ ]:

query_missing_attribute_overlap = """
SELECT
    CASE
        WHEN category_code IS NULL AND brand IS NULL
            THEN 'Both Missing'
        WHEN category_code IS NULL
            THEN 'Category Missing Only'
        WHEN brand IS NULL
            THEN 'Brand Missing Only'
        ELSE 'Both Available'
    END AS attribute_status,

    COUNT(*) AS purchase_events,
    SUM(price) AS purchase_event_gmv

FROM read_parquet(?)
WHERE event_type = 'purchase'
GROUP BY attribute_status
ORDER BY attribute_status
"""

missing_attribute_overlap = con.execute(
    query_missing_attribute_overlap,
    [OUTPUT_SESSION_FILE]
).df()

print("========== PURCHASE ATTRIBUTE OVERLAP ==========")
display(missing_attribute_overlap)


In [ ]:

query_brand_adjudication = """
WITH historical_conflicts AS (
    SELECT product_id
    FROM read_parquet(?)
    GROUP BY product_id
    HAVING COUNT(DISTINCT brand) > 1
),
purchase_brand_detail AS (
    SELECT
        e.product_id,
        COALESCE(e.brand, '[MISSING]') AS purchase_brand,
        COUNT(*) AS purchase_events,
        SUM(e.price) AS purchase_gmv,
        MIN(e.event_time) AS first_purchase,
        MAX(e.event_time) AS last_purchase
    FROM read_parquet(?) e
    INNER JOIN historical_conflicts h
        ON e.product_id = h.product_id
    WHERE e.event_type = 'purchase'
    GROUP BY
        e.product_id,
        COALESCE(e.brand, '[MISSING]')
),
product_summary AS (
    SELECT
        product_id,
        SUM(purchase_events) AS total_purchase_events,
        COUNT(*) FILTER (
            WHERE purchase_brand != '[MISSING]'
        ) AS non_missing_brand_count,
        COUNT(*) FILTER (
            WHERE purchase_brand = '[MISSING]'
        ) AS missing_brand_group_count,
        SUM(purchase_gmv) AS total_purchase_gmv,
        STRING_AGG(
            purchase_brand,
            ' | ' ORDER BY purchase_brand
        ) AS purchase_brand_profile
    FROM purchase_brand_detail
    GROUP BY product_id
)
SELECT
    *,
    CASE
        WHEN non_missing_brand_count > 1
            THEN 'Multiple Purchase Brands'
        WHEN non_missing_brand_count = 1
             AND missing_brand_group_count > 0
            THEN 'One Brand + Missing'
        WHEN non_missing_brand_count = 1
            THEN 'Single Purchase Brand'
        ELSE 'All Purchase Brands Missing'
    END AS adjudication_status
FROM product_summary
ORDER BY
    CASE
        WHEN non_missing_brand_count > 1 THEN 1
        WHEN non_missing_brand_count = 1
             AND missing_brand_group_count > 0 THEN 2
        WHEN non_missing_brand_count = 1 THEN 3
        ELSE 4
    END,
    total_purchase_gmv DESC
"""

brand_adjudication = con.execute(
    query_brand_adjudication,
    [OUTPUT_SESSION_FILE, OUTPUT_SESSION_FILE]
).df()

print("========== BRAND ADJUDICATION SUMMARY ==========")
display(
    brand_adjudication.groupby(
        "adjudication_status"
    ).agg(
        products=("product_id", "nunique"),
        purchase_events=("total_purchase_events", "sum"),
        purchase_gmv=("total_purchase_gmv", "sum")
    ).reset_index()
)

print("========== PRODUCT-LEVEL BRAND ADJUDICATION ==========")
display(brand_adjudication)


In [ ]:

print("========== BRAND-LEVEL PURCHASE DETAILS ==========")

brand_detail = con.execute("""
WITH historical_conflicts AS (
    SELECT product_id
    FROM read_parquet(?)
    GROUP BY product_id
    HAVING COUNT(DISTINCT brand) > 1
)
SELECT
    e.product_id,
    COALESCE(e.brand, '[MISSING]') AS purchase_brand,
    COUNT(*) AS purchase_events,
    ROUND(SUM(e.price), 2) AS purchase_gmv,
    MIN(e.event_time) AS first_purchase,
    MAX(e.event_time) AS last_purchase
FROM read_parquet(?) e
INNER JOIN historical_conflicts h
    ON e.product_id = h.product_id
WHERE e.event_type = 'purchase'
GROUP BY
    e.product_id,
    COALESCE(e.brand, '[MISSING]')
ORDER BY
    e.product_id,
    purchase_events DESC
""", [OUTPUT_SESSION_FILE, OUTPUT_SESSION_FILE]).df()

display(brand_detail)

brand_adjudication.to_csv(
    "Brand_Adjudication_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

brand_detail.to_csv(
    "Brand_Adjudication_Detail.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Adjudication files exported.")


In [ ]:

query_product_attribute_audit = """
WITH product_profile AS (
    SELECT
        product_id,
        COUNT(DISTINCT category_id) AS category_id_count,
        COUNT(DISTINCT category_code) AS category_code_count,
        COUNT(DISTINCT brand) AS historical_brand_count,

        STRING_AGG(
            DISTINCT CAST(category_id AS VARCHAR),
            ' | '
        ) AS category_ids,

        STRING_AGG(
            DISTINCT category_code,
            ' | '
        ) AS category_codes,

        STRING_AGG(
            DISTINCT brand,
            ' | '
        ) AS historical_brands,

        COUNT(*) AS total_event_rows,

        COUNT(*) FILTER (
            WHERE brand IS NULL
        ) AS missing_brand_event_rows,

        COUNT(*) FILTER (
            WHERE category_code IS NULL
        ) AS missing_category_event_rows

    FROM read_parquet(?)
    WHERE product_id IS NOT NULL
    GROUP BY product_id
),

purchase_profile AS (
    SELECT
        product_id,
        COUNT(*) AS purchase_events,

        COUNT(DISTINCT brand) AS purchase_brand_count,

        COUNT(*) FILTER (
            WHERE brand IS NULL
        ) AS missing_brand_purchases,

        STRING_AGG(
            DISTINCT brand,
            ' | '
        ) AS purchase_brands,

        ROUND(SUM(price), 2) AS purchase_gmv

    FROM read_parquet(?)
    WHERE event_type = 'purchase'
      AND product_id IS NOT NULL
    GROUP BY product_id
)

SELECT
    p.*,
    COALESCE(b.purchase_events, 0) AS purchase_events,
    COALESCE(b.purchase_brand_count, 0) AS purchase_brand_count,
    COALESCE(b.missing_brand_purchases, 0) AS missing_brand_purchases,
    b.purchase_brands,
    COALESCE(b.purchase_gmv, 0) AS purchase_gmv,

    CASE
        WHEN p.category_id_count > 1
          OR p.category_code_count > 1
            THEN 'Category Conflict'

        WHEN p.historical_brand_count > 1
            THEN 'Historical Brand Conflict'

        ELSE 'No Historical Attribute Conflict'
    END AS historical_attribute_status,

    CASE
        WHEN p.historical_brand_count <= 1
            THEN 'No Historical Brand Conflict'

        WHEN COALESCE(b.purchase_events, 0) = 0
            THEN 'No Purchase Evidence'

        WHEN b.purchase_brand_count > 1
            THEN 'Multiple Purchase Brands'

        WHEN b.purchase_brand_count = 1
             AND b.missing_brand_purchases > 0
            THEN 'One Brand + Missing'

        WHEN b.purchase_brand_count = 1
            THEN 'Single Purchase Brand'

        ELSE 'All Purchase Brands Missing'
    END AS purchase_brand_status

FROM product_profile p
LEFT JOIN purchase_profile b
    ON p.product_id = b.product_id
"""

product_attribute_audit = con.execute(
    query_product_attribute_audit,
    [OUTPUT_SESSION_FILE, OUTPUT_SESSION_FILE]
).df()

print("========== PRODUCT ATTRIBUTE AUDIT ==========")
print("Total products:", len(product_attribute_audit))
print(
    "Unique product IDs:",
    product_attribute_audit["product_id"].nunique()
)

print("\n========== HISTORICAL ATTRIBUTE STATUS ==========")
display(
    product_attribute_audit[
        "historical_attribute_status"
    ].value_counts().rename_axis(
        "status"
    ).reset_index(name="products")
)

print("\n========== PURCHASE BRAND STATUS ==========")
display(
    product_attribute_audit[
        "purchase_brand_status"
    ].value_counts().rename_axis(
        "status"
    ).reset_index(name="products")
)

print("\n========== CONFLICTING PRODUCT SAMPLE ==========")
display(
    product_attribute_audit[
        product_attribute_audit[
            "historical_attribute_status"
        ] == "Historical Brand Conflict"
    ].head(30)
)

product_attribute_audit.to_csv(
    "Product_Attribute_Audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nExported: Product_Attribute_Audit.csv")


In [ ]:

# Phase 1.6.3 — Product Dimension Readiness Audit

print("========== PRODUCT DIMENSION READINESS ==========")

# 1. Product-level attribute coverage
product_coverage = pd.DataFrame({
    "metric": [
        "Total Products",
        "Products with Category Code",
        "Products Missing Category Code",
        "Products with Brand",
        "Products Missing Brand",
        "Products with Historical Brand Conflict",
        "Products with Purchase Events",
        "Products without Purchase Events"
    ],
    "count": [
        len(product_attribute_audit),
        (product_attribute_audit["category_code_count"] > 0).sum(),
        (product_attribute_audit["category_code_count"] == 0).sum(),
        (product_attribute_audit["historical_brand_count"] > 0).sum(),
        (product_attribute_audit["historical_brand_count"] == 0).sum(),
        (product_attribute_audit["historical_brand_count"] > 1).sum(),
        (product_attribute_audit["purchase_events"] > 0).sum(),
        (product_attribute_audit["purchase_events"] == 0).sum()
    ]
})

product_coverage["percentage"] = (
    product_coverage["count"] /
    len(product_attribute_audit) * 100
).round(4)

display(product_coverage)

# 2. Purchase-event product mapping coverage
purchase_mapping = con.execute("""
SELECT
    COUNT(*) AS total_purchase_events,
    COUNT(product_id) AS purchases_with_product_id,
    COUNT(*) FILTER (
        WHERE product_id IS NULL
    ) AS purchases_without_product_id,
    COUNT(DISTINCT product_id) AS unique_purchase_products,
    ROUND(SUM(price), 2) AS total_purchase_gmv
FROM read_parquet(?)
WHERE event_type = 'purchase'
""", [OUTPUT_SESSION_FILE]).df()

print("\n========== PURCHASE PRODUCT MAPPING ==========")
display(purchase_mapping)

# 3. Check whether every purchase product exists in audit table
unmatched_products = con.execute("""
SELECT
    COUNT(*) AS unmatched_purchase_events,
    COUNT(DISTINCT e.product_id) AS unmatched_product_ids,
    ROUND(SUM(e.price), 2) AS unmatched_purchase_gmv
FROM read_parquet(?) e
LEFT JOIN (
    SELECT DISTINCT product_id
    FROM read_parquet(?)
) p
    ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
  AND e.product_id IS NOT NULL
  AND p.product_id IS NULL
""", [OUTPUT_SESSION_FILE, OUTPUT_SESSION_FILE]).df()

print("\n========== UNMATCHED PURCHASE PRODUCTS ==========")
display(unmatched_products)


In [ ]:

query_user_dimension = """
SELECT
    user_id,

    COUNT(*) AS total_events,

    COUNT(*) FILTER (
        WHERE event_type = 'view'
    ) AS view_events,

    COUNT(*) FILTER (
        WHERE event_type = 'cart'
    ) AS cart_events,

    COUNT(*) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_events,

    COUNT(DISTINCT user_session) AS user_session_pairs,

    SUM(
        CASE
            WHEN event_type = 'purchase' THEN price
            ELSE 0
        END
    ) AS total_spent,

    MIN(event_time) AS first_activity,
    MAX(event_time) AS last_activity

FROM read_parquet(?)
WHERE user_id IS NOT NULL
GROUP BY user_id
"""

user_dimension = con.execute(
    query_user_dimension,
    [OUTPUT_SESSION_FILE]
).df()

print("========== USER DIMENSION ==========")
print("Rows:", len(user_dimension))
print("Unique user IDs:", user_dimension["user_id"].nunique())
print("Duplicate user IDs:", user_dimension["user_id"].duplicated().sum())

display(user_dimension.head(10))


In [ ]:

user_dimension["purchase_status"] = pd.cut(
    user_dimension["purchase_events"],
    bins=[-1, 0, 1, float("inf")],
    labels=[
        "No Purchase",
        "Single Purchase",
        "Repeat Purchase"
    ]
)

print("========== USER PURCHASE DISTRIBUTION ==========")

purchase_distribution = (
    user_dimension["purchase_status"]
    .value_counts()
    .reindex([
        "No Purchase",
        "Single Purchase",
        "Repeat Purchase"
    ])
    .reset_index()
)

purchase_distribution.columns = [
    "purchase_status",
    "users"
]

purchase_distribution["percentage"] = (
    purchase_distribution["users"]
    / len(user_dimension) * 100
).round(4)

display(purchase_distribution)

print("========== USER SPEND SUMMARY ==========")

display(
    user_dimension[
        user_dimension["purchase_events"] > 0
    ]["total_spent"].describe()
)

print("========== USER DIMENSION TOTALS ==========")

display(
    user_dimension[
        [
            "total_events",
            "view_events",
            "cart_events",
            "purchase_events",
            "user_session_pairs",
            "total_spent"
        ]
    ].sum().to_frame("total")
)


In [ ]:

global_user_check = con.execute("""
SELECT
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(*) AS total_events,

    COUNT(*) FILTER (
        WHERE event_type = 'view'
    ) AS view_events,

    COUNT(*) FILTER (
        WHERE event_type = 'cart'
    ) AS cart_events,

    COUNT(*) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_events,

    COUNT(DISTINCT user_id || '|' || user_session)
        AS user_session_pairs,

    SUM(
        CASE
            WHEN event_type = 'purchase' THEN price
            ELSE 0
        END
    ) AS purchase_event_gmv,

    COUNT(*) FILTER (
        WHERE user_id IS NULL
    ) AS null_user_events

FROM read_parquet(?)
""", [OUTPUT_SESSION_FILE]).df()

print("========== GLOBAL USER RECONCILIATION ==========")
display(global_user_check)

print("========== USER DIMENSION RECONCILIATION ==========")

reconciliation = pd.DataFrame({
    "metric": [
        "Unique Users",
        "Total Events",
        "View Events",
        "Cart Events",
        "Purchase Events",
        "User-Session Pairs",
        "Purchase-event GMV"
    ],
    "user_dimension": [
        user_dimension["user_id"].nunique(),
        user_dimension["total_events"].sum(),
        user_dimension["view_events"].sum(),
        user_dimension["cart_events"].sum(),
        user_dimension["purchase_events"].sum(),
        user_dimension["user_session_pairs"].sum(),
        user_dimension["total_spent"].sum()
    ],
    "global_events": [
        global_user_check.iloc[0]["unique_users"],
        global_user_check.iloc[0]["total_events"],
        global_user_check.iloc[0]["view_events"],
        global_user_check.iloc[0]["cart_events"],
        global_user_check.iloc[0]["purchase_events"],
        global_user_check.iloc[0]["user_session_pairs"],
        global_user_check.iloc[0]["purchase_event_gmv"]
    ]
})

reconciliation["difference"] = (
    reconciliation["user_dimension"]
    - reconciliation["global_events"]
)

display(reconciliation)


In [ ]:

query_funnel_audit = """
WITH user_first_view AS (
    SELECT
        user_id,
        MIN(event_time) AS first_view
    FROM read_parquet(?)
    WHERE event_type = 'view'
      AND user_id IS NOT NULL
    GROUP BY user_id
),

user_first_cart AS (
    SELECT
        e.user_id,
        MIN(e.event_time) AS first_cart
    FROM read_parquet(?) e
    INNER JOIN user_first_view v
        ON e.user_id = v.user_id
    WHERE e.event_type = 'cart'
      AND e.event_time > v.first_view
    GROUP BY e.user_id
),

user_first_purchase AS (
    SELECT
        e.user_id,
        MIN(e.event_time) AS first_purchase
    FROM read_parquet(?) e
    INNER JOIN user_first_cart c
        ON e.user_id = c.user_id
    WHERE e.event_type = 'purchase'
      AND e.event_time > c.first_cart
    GROUP BY e.user_id
),

user_funnel AS (
    SELECT
        COUNT(*) AS view_users,
        COUNT(c.user_id) AS cart_after_view_users,
        COUNT(p.user_id) AS purchase_after_cart_users
    FROM user_first_view v
    LEFT JOIN user_first_cart c
        ON v.user_id = c.user_id
    LEFT JOIN user_first_purchase p
        ON v.user_id = p.user_id
),

session_first_view AS (
    SELECT
        user_id,
        user_session,
        MIN(event_time) AS first_view
    FROM read_parquet(?)
    WHERE event_type = 'view'
      AND user_id IS NOT NULL
      AND user_session IS NOT NULL
    GROUP BY user_id, user_session
),

session_first_cart AS (
    SELECT
        e.user_id,
        e.user_session,
        MIN(e.event_time) AS first_cart
    FROM read_parquet(?) e
    INNER JOIN session_first_view v
        ON e.user_id = v.user_id
       AND e.user_session = v.user_session
    WHERE e.event_type = 'cart'
      AND e.event_time > v.first_view
    GROUP BY e.user_id, e.user_session
),

session_first_purchase AS (
    SELECT
        e.user_id,
        e.user_session,
        MIN(e.event_time) AS first_purchase
    FROM read_parquet(?) e
    INNER JOIN session_first_cart c
        ON e.user_id = c.user_id
       AND e.user_session = c.user_session
    WHERE e.event_type = 'purchase'
      AND e.event_time > c.first_cart
    GROUP BY e.user_id, e.user_session
),

session_funnel AS (
    SELECT
        COUNT(*) AS view_sessions,
        COUNT(c.user_id) AS cart_after_view_sessions,
        COUNT(p.user_id) AS purchase_after_cart_sessions
    FROM session_first_view v
    LEFT JOIN session_first_cart c
        ON v.user_id = c.user_id
       AND v.user_session = c.user_session
    LEFT JOIN session_first_purchase p
        ON v.user_id = p.user_id
       AND v.user_session = p.user_session
)

SELECT
    'User Funnel' AS funnel_type,
    view_users AS stage_1_view,
    cart_after_view_users AS stage_2_cart,
    purchase_after_cart_users AS stage_3_purchase
FROM user_funnel

UNION ALL

SELECT
    'Session Funnel',
    view_sessions,
    cart_after_view_sessions,
    purchase_after_cart_sessions
FROM session_funnel
"""

funnel_result = con.execute(
    query_funnel_audit,
    [
        OUTPUT_SESSION_FILE,
        OUTPUT_SESSION_FILE,
        OUTPUT_SESSION_FILE,
        OUTPUT_SESSION_FILE,
        OUTPUT_SESSION_FILE,
        OUTPUT_SESSION_FILE
    ]
).df()

print("========== STRICT SEQUENTIAL FUNNEL ==========")
display(funnel_result)

funnel_result["view_to_cart_rate"] = (
    funnel_result["stage_2_cart"]
    / funnel_result["stage_1_view"] * 100
).round(4)

funnel_result["cart_to_purchase_rate"] = (
    funnel_result["stage_3_purchase"]
    / funnel_result["stage_2_cart"] * 100
).round(4)

funnel_result["view_to_purchase_rate"] = (
    funnel_result["stage_3_purchase"]
    / funnel_result["stage_1_view"] * 100
).round(4)

print("========== FUNNEL CONVERSION RATES (%) ==========")
display(funnel_result)


In [ ]:

query_retention = """
WITH events AS (
    SELECT
        user_id,
        CAST(event_time AS DATE) AS activity_date
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
),

user_cohort AS (
    SELECT
        user_id,
        MIN(activity_date) AS cohort_date
    FROM events
    GROUP BY user_id
),

active_days AS (
    SELECT DISTINCT
        user_id,
        activity_date
    FROM events
),

data_boundary AS (
    SELECT MAX(activity_date) AS max_date
    FROM events
),

cohort_activity AS (
    SELECT
        c.user_id,
        c.cohort_date,
        MAX(CASE
            WHEN a.activity_date =
                 c.cohort_date + INTERVAL 1 DAY
            THEN 1 ELSE 0
        END) AS day_1_returned,

        MAX(CASE
            WHEN a.activity_date =
                 c.cohort_date + INTERVAL 7 DAY
            THEN 1 ELSE 0
        END) AS day_7_returned

    FROM user_cohort c
    LEFT JOIN active_days a
        ON c.user_id = a.user_id
       AND a.activity_date > c.cohort_date
    GROUP BY c.user_id, c.cohort_date
),

cohort_summary AS (
    SELECT
        ca.cohort_date,
        COUNT(*) AS cohort_users,

        SUM(CASE
            WHEN ca.cohort_date + INTERVAL 1 DAY <= d.max_date
            THEN 1 ELSE 0
        END) AS day_1_eligible_users,

        SUM(CASE
            WHEN ca.cohort_date + INTERVAL 1 DAY <= d.max_date
            THEN ca.day_1_returned ELSE 0
        END) AS day_1_returned_users,

        SUM(CASE
            WHEN ca.cohort_date + INTERVAL 7 DAY <= d.max_date
            THEN 1 ELSE 0
        END) AS day_7_eligible_users,

        SUM(CASE
            WHEN ca.cohort_date + INTERVAL 7 DAY <= d.max_date
            THEN ca.day_7_returned ELSE 0
        END) AS day_7_returned_users

    FROM cohort_activity ca
    CROSS JOIN data_boundary d
    GROUP BY ca.cohort_date
)

SELECT
    *,
    ROUND(
        100.0 * day_1_returned_users /
        NULLIF(day_1_eligible_users, 0), 4
    ) AS day_1_retention_pct,

    ROUND(
        100.0 * day_7_returned_users /
        NULLIF(day_7_eligible_users, 0), 4
    ) AS day_7_retention_pct

FROM cohort_summary
ORDER BY cohort_date
"""

retention_result = con.execute(
    query_retention,
    [OUTPUT_SESSION_FILE]
).df()

print("========== COHORT RETENTION BY DATE ==========")
display(retention_result)

# Overall retention: only users with complete observation windows
day1_eligible = retention_result["day_1_eligible_users"].sum()
day1_returned = retention_result["day_1_returned_users"].sum()

day7_eligible = retention_result["day_7_eligible_users"].sum()
day7_returned = retention_result["day_7_returned_users"].sum()

retention_overall = pd.DataFrame([
    {
        "metric": "Day-1 Retention",
        "eligible_users": int(day1_eligible),
        "returned_users": int(day1_returned),
        "retention_pct": round(
            day1_returned / day1_eligible * 100, 4
        ) if day1_eligible else None
    },
    {
        "metric": "Day-7 Retention",
        "eligible_users": int(day7_eligible),
        "returned_users": int(day7_returned),
        "retention_pct": round(
            day7_returned / day7_eligible * 100, 4
        ) if day7_eligible else None
    }
])

print("========== OVERALL RETENTION ==========")
display(retention_overall)

print("========== DATA OBSERVATION BOUNDARY ==========")
print("Maximum observed activity date:",
      con.execute(
          "SELECT MAX(CAST(event_time AS DATE)) "
          "FROM read_parquet(?)",
          [OUTPUT_SESSION_FILE]
      ).fetchone()[0])


In [ ]:

# Phase 1.8.3 — Retention Consistency Audit

print("========== RETENTION COHORT RECONCILIATION ==========")

print("Total cohort users:",
      retention_result["cohort_users"].sum())

print("Global unique users:",
      3022290)

print("Cohort user difference:",
      retention_result["cohort_users"].sum() - 3022290)

print("\n========== ELIGIBILITY VALIDATION ==========")

print(
    "Day-1 returned <= eligible:",
    (
        retention_result["day_1_returned_users"]
        <= retention_result["day_1_eligible_users"]
    ).all()
)

print(
    "Day-7 returned <= eligible:",
    (
        retention_result["day_7_returned_users"]
        <= retention_result["day_7_eligible_users"]
    ).all()
)

print("\n========== COHORT DATE VALIDATION ==========")

print("Number of cohort dates:",
      retention_result["cohort_date"].nunique())

print("Duplicate cohort dates:",
      retention_result["cohort_date"].duplicated().sum())

print("Day-1 ineligible cohorts:",
      (retention_result["day_1_eligible_users"] == 0).sum())

print("Day-7 ineligible cohorts:",
      (retention_result["day_7_eligible_users"] == 0).sum())

print("\n========== RETENTION RATE BOUNDARY ==========")

print(
    "Day-1 rates within 0-100:",
    retention_result["day_1_retention_pct"]
    .dropna().between(0, 100).all()
)

print(
    "Day-7 rates within 0-100:",
    retention_result["day_7_retention_pct"]
    .dropna().between(0, 100).all()
)


In [ ]:

from pathlib import Path
import pandas as pd

# 如果此前已有 retention.csv 路径变量，可直接替换这里
LEGACY_RETENTION_PATH = Path(
    r"D:\PowerBI_Project\retention.csv"
)

print("========== LEGACY RETENTION FILE ==========")
print("Path:", LEGACY_RETENTION_PATH)
print("Exists:", LEGACY_RETENTION_PATH.exists())

if LEGACY_RETENTION_PATH.exists():
    legacy_retention = pd.read_csv(LEGACY_RETENTION_PATH)

    print("\n========== LEGACY COLUMNS ==========")
    print(legacy_retention.columns.tolist())

    print("\n========== LEGACY SAMPLE ==========")
    display(legacy_retention.head(15))

    print("\n========== LEGACY DATA SUMMARY ==========")
    print("Rows:", len(legacy_retention))
    print("Missing values:")
    display(legacy_retention.isna().sum())

else:
    print(
        "File not found. Please update "
        "LEGACY_RETENTION_PATH to the actual CSV location."
    )


In [ ]:

# Phase 1.8.5 — Legacy vs V3 Reconciliation

legacy = legacy_retention.copy()
legacy["fir_date"] = pd.to_datetime(legacy["fir_date"]).dt.date

v3 = retention_result.copy()
v3["cohort_date"] = pd.to_datetime(v3["cohort_date"]).dt.date

comparison = legacy.merge(
    v3,
    left_on="fir_date",
    right_on="cohort_date",
    how="outer"
)

comparison["legacy_calculated_rate"] = (
    comparison["retained_users"] /
    comparison["new_user_cnt"]
)

comparison["legacy_rate_check"] = (
    comparison["retention_rate"] -
    comparison["legacy_calculated_rate"]
)

comparison["cohort_user_difference"] = (
    comparison["cohort_users"] -
    comparison["new_user_cnt"]
)

comparison["retained_user_difference"] = (
    comparison["day_1_returned_users"] -
    comparison["retained_users"]
)

comparison["retention_rate_difference_pp"] = (
    comparison["day_1_retention_pct"] -
    comparison["retention_rate"] * 100
)

columns_to_show = [
    "fir_date",
    "new_user_cnt",
    "cohort_users",
    "cohort_user_difference",
    "retained_users",
    "day_1_returned_users",
    "retained_user_difference",
    "retention_rate",
    "legacy_calculated_rate",
    "day_1_retention_pct",
    "retention_rate_difference_pp"
]

print("========== LEGACY VS V3 COMPARISON ==========")
display(comparison[columns_to_show])

print("\n========== LEGACY RATE VALIDATION ==========")
print(
    "Legacy stored rate matches retained/new users:",
    comparison["legacy_rate_check"].abs().max() < 0.0001
)

print("\n========== LEGACY DATE COVERAGE ==========")
print("Legacy rows:", len(legacy))
print("V3 cohort rows:", len(v3))
print("Legacy dates missing from V3:",
      comparison["cohort_users"].isna().sum())
print("V3 dates missing from Legacy:",
      comparison["new_user_cnt"].isna().sum())


In [ ]:

import json
from pathlib import Path

PROJECT_DIR = Path(r"D:\PowerBI_Project")

KEYWORDS = [
    "retention.csv",
    "retained_users",
    "retention_rate",
    "new_user_cnt",
    "fir_date"
]

print("========== SEARCHING LEGACY RETENTION CODE ==========")

notebooks = list(PROJECT_DIR.glob("*.ipynb"))

print("Notebooks found:", len(notebooks))

for notebook_path in notebooks:
    try:
        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        matched_cells = []

        for i, cell in enumerate(notebook.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            matched_keywords = [
                keyword for keyword in KEYWORDS
                if keyword.lower() in source.lower()
            ]

            if matched_keywords:
                matched_cells.append(
                    (i, matched_keywords, source)
                )

        if matched_cells:
            print("\n" + "=" * 70)
            print("NOTEBOOK:", notebook_path.name)

            for cell_index, keywords, source in matched_cells:
                print("\n--- CELL", cell_index, "---")
                print("Matched:", keywords)
                print(source[:8000])

    except Exception as e:
        print("Could not read:", notebook_path.name, str(e))


In [ ]:

import json
from pathlib import Path

PROJECT_DIR = Path(r"D:\PowerBI_Project")

SEARCH_TERMS = [
    "retention",
    "cohort",
    "retained",
    "returning",
    "first_activity",
    "activity_date",
    "fir_date",
    "new_user",
    "to_csv",
    "shift(",
    "groupby(",
    "next_day",
    "date_diff",
    "timedelta",
    "user_id"
]

print("========== RECURSIVE NOTEBOOK SEARCH ==========")

notebook_files = list(PROJECT_DIR.rglob("*.ipynb"))

print("Notebook files found:", len(notebook_files))

for path in notebook_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)

        print("\n" + "=" * 80)
        print("NOTEBOOK:", path)

        for i, cell in enumerate(nb.get("cells", [])):
            source = "".join(cell.get("source", []))

            if not source.strip():
                continue

            matched = [
                term for term in SEARCH_TERMS
                if term.lower() in source.lower()
            ]

            # Exclude the two newly added audit cells
            if matched and i not in [57, 58]:
                print("\n--- CELL", i, "---")
                print("Matched keywords:", matched)
                print(source[:5000])

    except Exception as e:
        print("ERROR:", path, str(e))

print("\n========== RECURSIVE SCRIPT SEARCH ==========")

script_extensions = ["*.py", "*.sql"]

for pattern in script_extensions:
    for path in PROJECT_DIR.rglob(pattern):
        try:
            source = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            matched = [
                term for term in SEARCH_TERMS
                if term.lower() in source.lower()
            ]

            if matched:
                print("\nFILE:", path)
                print("Matched keywords:", matched)
                print(source[:5000])

        except Exception as e:
            print("ERROR:", path, str(e))


In [ ]:

# Phase 1.8.6 — V3 Retention Logic Validation

import duckdb
import pandas as pd

con = duckdb.connect()

OUTPUT_SESSION_FILE = (
    r"D:\PowerBI_Project\ecommerce_session_ready.parquet"
)

query = """
WITH events AS (
    SELECT
        user_id,
        CAST(event_time AS DATE) AS activity_date
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
),

first_activity AS (
    SELECT
        user_id,
        MIN(activity_date) AS cohort_date
    FROM events
    GROUP BY user_id
),

user_activity AS (
    SELECT DISTINCT
        user_id,
        activity_date
    FROM events
),

manual_check AS (
    SELECT
        c.user_id,
        c.cohort_date,

        MAX(CASE
            WHEN a.activity_date = c.cohort_date + INTERVAL 1 DAY
            THEN 1 ELSE 0
        END) AS returned_day_1,

        MAX(CASE
            WHEN a.activity_date = c.cohort_date + INTERVAL 7 DAY
            THEN 1 ELSE 0
        END) AS returned_day_7

    FROM first_activity c
    LEFT JOIN user_activity a
        ON c.user_id = a.user_id
       AND a.activity_date > c.cohort_date

    GROUP BY c.user_id, c.cohort_date
)

SELECT
    cohort_date,
    COUNT(*) AS cohort_users,
    SUM(returned_day_1) AS day_1_returned,
    SUM(returned_day_7) AS day_7_returned
FROM manual_check
GROUP BY cohort_date
ORDER BY cohort_date
"""

manual_result = con.execute(
    query,
    [OUTPUT_SESSION_FILE]
).df()

print("========== INDEPENDENT RETENTION CHECK ==========")
display(manual_result)

print("\nTotal cohort users:",
      manual_result["cohort_users"].sum())

print("Day-1 returned users:",
      manual_result["day_1_returned"].sum())

print("Day-7 returned users:",
      manual_result["day_7_returned"].sum())


In [ ]:

# Phase 1.8.6.1 — Retention Eligibility Verification

query_eligibility = """
WITH events AS (
    SELECT
        user_id,
        CAST(event_time AS DATE) AS activity_date
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
),

user_cohort AS (
    SELECT
        user_id,
        MIN(activity_date) AS cohort_date
    FROM events
    GROUP BY user_id
),

data_boundary AS (
    SELECT MAX(activity_date) AS max_date
    FROM events
)

SELECT
    MAX(max_date) AS max_observation_date,

    COUNT(*) AS total_cohort_users,

    COUNT(*) FILTER (
        WHERE cohort_date + INTERVAL 1 DAY <= max_date
    ) AS day_1_eligible_users,

    COUNT(*) FILTER (
        WHERE cohort_date + INTERVAL 7 DAY <= max_date
    ) AS day_7_eligible_users

FROM user_cohort
CROSS JOIN data_boundary
"""

eligibility_result = con.execute(
    query_eligibility,
    [OUTPUT_SESSION_FILE]
).df()

display(eligibility_result)

day1_eligible_check = int(
    eligibility_result.iloc[0]["day_1_eligible_users"]
)

day7_eligible_check = int(
    eligibility_result.iloc[0]["day_7_eligible_users"]
)

day1_returned_check = 315853
day7_returned_check = 163206

print("\n========== FINAL RETENTION RATES ==========")

print(
    "Day-1:",
    round(day1_returned_check / day1_eligible_check * 100, 4),
    "%"
)

print(
    "Day-7:",
    round(day7_returned_check / day7_eligible_check * 100, 4),
    "%"
)


In [ ]:

# Phase 1.8.7.4 — Retention Power BI Export

# Table A: Cohort-level retention
retention_cohort_export = retention_result.copy()

# Add observation boundary
retention_cohort_export["max_observation_date"] = pd.to_datetime(
    "2019-10-31"
)

# Eligibility status
retention_cohort_export["day_1_status"] = (
    retention_cohort_export["day_1_eligible_users"]
    .gt(0)
    .map({True: "Eligible", False: "Not Eligible"})
)

retention_cohort_export["day_7_status"] = (
    retention_cohort_export["day_7_eligible_users"]
    .gt(0)
    .map({True: "Eligible", False: "Not Eligible"})
)

# Use NULL for returned users when the cohort is ineligible
retention_cohort_export.loc[
    retention_cohort_export["day_1_status"] == "Not Eligible",
    "day_1_returned_users"
] = pd.NA

retention_cohort_export.loc[
    retention_cohort_export["day_7_status"] == "Not Eligible",
    "day_7_returned_users"
] = pd.NA

# Table B: Overall retention
retention_overall_export = retention_overall.copy()

# Export
retention_cohort_export.to_csv(
    "Retention_Cohort.csv",
    index=False
)

retention_overall_export.to_csv(
    "Retention_Overall.csv",
    index=False
)

# Validation
print("Cohort rows:", len(retention_cohort_export))
print("Overall rows:", len(retention_overall_export))

print("\nDay-1 eligible cohorts:",
      (retention_cohort_export["day_1_status"] == "Eligible").sum())

print("Day-1 ineligible cohorts:",
      (retention_cohort_export["day_1_status"] == "Not Eligible").sum())

print("Day-7 eligible cohorts:",
      (retention_cohort_export["day_7_status"] == "Eligible").sum())

print("Day-7 ineligible cohorts:",
      (retention_cohort_export["day_7_status"] == "Not Eligible").sum())

print("\nExport completed.")
display(retention_overall_export)


In [ ]:

# Phase 1.8.7.6 — CSV Integrity Audit

import pandas as pd
from pathlib import Path

cohort_path = Path("Retention_Cohort.csv")
overall_path = Path("Retention_Overall.csv")

assert cohort_path.exists(), "Retention_Cohort.csv not found"
assert overall_path.exists(), "Retention_Overall.csv not found"

cohort = pd.read_csv(cohort_path)
overall = pd.read_csv(overall_path)

# 1. Schema validation
required_cohort_columns = {
    "cohort_date",
    "cohort_users",
    "day_1_eligible_users",
    "day_1_returned_users",
    "day_1_retention_pct",
    "day_1_status",
    "day_7_eligible_users",
    "day_7_returned_users",
    "day_7_retention_pct",
    "day_7_status",
    "max_observation_date"
}

required_overall_columns = {
    "metric",
    "eligible_users",
    "returned_users",
    "retention_pct"
}

assert required_cohort_columns.issubset(cohort.columns)
assert required_overall_columns.issubset(overall.columns)

# 2. Row count
assert len(cohort) == 31
assert len(overall) == 2

# 3. Eligibility and NULL validation
assert (cohort["day_1_status"] == "Eligible").sum() == 30
assert (cohort["day_1_status"] == "Not Eligible").sum() == 1

assert (cohort["day_7_status"] == "Eligible").sum() == 24
assert (cohort["day_7_status"] == "Not Eligible").sum() == 7

assert cohort.loc[
    cohort["day_1_status"] == "Not Eligible",
    "day_1_returned_users"
].isna().all()

assert cohort.loc[
    cohort["day_7_status"] == "Not Eligible",
    "day_7_returned_users"
].isna().all()

assert cohort.loc[
    cohort["day_1_status"] == "Not Eligible",
    "day_1_retention_pct"
].isna().all()

assert cohort.loc[
    cohort["day_7_status"] == "Not Eligible",
    "day_7_retention_pct"
].isna().all()

# 4. Overall reconciliation
day1 = overall.loc[
    overall["metric"] == "Day-1 Retention"
].iloc[0]

day7 = overall.loc[
    overall["metric"] == "Day-7 Retention"
].iloc[0]

assert int(cohort["day_1_eligible_users"].sum()) == int(day1["eligible_users"])
assert int(cohort["day_1_returned_users"].sum()) == int(day1["returned_users"])

assert int(cohort["day_7_eligible_users"].sum()) == int(day7["eligible_users"])
assert int(cohort["day_7_returned_users"].sum()) == int(day7["returned_users"])

# 5. Date and uniqueness
cohort["cohort_date"] = pd.to_datetime(cohort["cohort_date"])

assert cohort["cohort_date"].is_unique
assert cohort["cohort_date"].min() == pd.Timestamp("2019-10-01")
assert cohort["cohort_date"].max() == pd.Timestamp("2019-10-31")

print("========== CSV INTEGRITY AUDIT ==========")
print("Schema validation: PASS")
print("Row count validation: PASS")
print("Eligibility validation: PASS")
print("NULL handling validation: PASS")
print("Overall reconciliation: PASS")
print("Cohort date validation: PASS")

print("\n========== FILE INFORMATION ==========")
print("Cohort file:", cohort_path.resolve())
print("Overall file:", overall_path.resolve())
print("Cohort rows:", len(cohort))
print("Overall rows:", len(overall))


In [ ]:

# Phase 1.8.8.1 — Strict Funnel Result Validation

import pandas as pd

funnel_check = funnel_result.copy()

print("========== FUNNEL RESULT ==========")
display(funnel_check)

# 1. Required columns
required_columns = [
    "funnel_type",
    "stage_1_view",
    "stage_2_cart",
    "stage_3_purchase",
    "view_to_cart_rate",
    "cart_to_purchase_rate",
    "view_to_purchase_rate"
]

missing_columns = [
    c for c in required_columns
    if c not in funnel_check.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

# 2. Funnel types
assert set(funnel_check["funnel_type"]) == {
    "User Funnel",
    "Session Funnel"
}

assert len(funnel_check) == 2

# 3. Counts must be non-negative
count_columns = [
    "stage_1_view",
    "stage_2_cart",
    "stage_3_purchase"
]

assert (funnel_check[count_columns] >= 0).all().all()

# 4. Sequential funnel monotonicity
assert (
    funnel_check["stage_1_view"]
    >= funnel_check["stage_2_cart"]
).all()

assert (
    funnel_check["stage_2_cart"]
    >= funnel_check["stage_3_purchase"]
).all()

# 5. Independently recalculate conversion rates
funnel_check["check_view_to_cart"] = (
    funnel_check["stage_2_cart"]
    / funnel_check["stage_1_view"] * 100
).round(4)

funnel_check["check_cart_to_purchase"] = (
    funnel_check["stage_3_purchase"]
    / funnel_check["stage_2_cart"] * 100
).round(4)

funnel_check["check_view_to_purchase"] = (
    funnel_check["stage_3_purchase"]
    / funnel_check["stage_1_view"] * 100
).round(4)

assert (
    funnel_check["view_to_cart_rate"]
    == funnel_check["check_view_to_cart"]
).all()

assert (
    funnel_check["cart_to_purchase_rate"]
    == funnel_check["check_cart_to_purchase"]
).all()

assert (
    funnel_check["view_to_purchase_rate"]
    == funnel_check["check_view_to_purchase"]
).all()

# 6. Final audit
print("\n========== FUNNEL VALIDATION ==========")
print("Required columns: PASS")
print("Funnel type validation: PASS")
print("Non-negative counts: PASS")
print("Sequential stage validation: PASS")
print("Conversion rate recalculation: PASS")

print("\n========== FINAL FUNNEL METRICS ==========")
display(
    funnel_check[
        [
            "funnel_type",
            "stage_1_view",
            "stage_2_cart",
            "stage_3_purchase",
            "view_to_cart_rate",
            "cart_to_purchase_rate",
            "view_to_purchase_rate"
        ]
    ]
)


In [ ]:

# Phase 1.8.8.2 — Independent Funnel Population Audit

query_funnel_population = """
WITH base AS (
    SELECT
        user_id,
        user_session,
        event_type
    FROM read_parquet(?)
),

global_users AS (
    SELECT COUNT(DISTINCT user_id) AS total_users
    FROM base
    WHERE user_id IS NOT NULL
),

view_users AS (
    SELECT COUNT(DISTINCT user_id) AS view_users
    FROM base
    WHERE event_type = 'view'
      AND user_id IS NOT NULL
),

view_session_pairs AS (
    SELECT COUNT(*) AS view_sessions
    FROM (
        SELECT user_id, user_session
        FROM base
        WHERE event_type = 'view'
          AND user_id IS NOT NULL
          AND user_session IS NOT NULL
        GROUP BY user_id, user_session
    )
),

view_null_sessions AS (
    SELECT COUNT(*) AS view_events_with_null_session
    FROM base
    WHERE event_type = 'view'
      AND user_id IS NOT NULL
      AND user_session IS NULL
),

view_users_without_session AS (
    SELECT COUNT(DISTINCT user_id) AS users_without_valid_view_session
    FROM base
    WHERE event_type = 'view'
      AND user_id IS NOT NULL
      AND user_session IS NULL
)

SELECT
    g.total_users,
    v.view_users,
    g.total_users - v.view_users AS users_without_view,
    s.view_sessions,
    ROUND(
        1.0 * s.view_sessions / NULLIF(v.view_users, 0), 4
    ) AS view_sessions_per_view_user,
    n.view_events_with_null_session,
    u.users_without_valid_view_session

FROM global_users g
CROSS JOIN view_users v
CROSS JOIN view_session_pairs s
CROSS JOIN view_null_sessions n
CROSS JOIN view_users_without_session u
"""

funnel_population_check = con.execute(
    query_funnel_population,
    [OUTPUT_SESSION_FILE]
).df()

display(funnel_population_check)


In [ ]:

# Phase 1.8.8.3 — Independent Funnel Stage Reconciliation

query_funnel_stage_check = """
WITH base AS (
    SELECT
        user_id,
        user_session,
        event_time,
        event_type
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
),

-- User-level: first view for each user
user_view AS (
    SELECT
        user_id,
        MIN(event_time) AS first_view
    FROM base
    WHERE event_type = 'view'
    GROUP BY user_id
),

-- User-level: first cart after first view
user_cart AS (
    SELECT
        v.user_id,
        v.first_view,
        MIN(b.event_time) AS first_cart
    FROM user_view v
    JOIN base b
      ON b.user_id = v.user_id
     AND b.event_type = 'cart'
     AND b.event_time > v.first_view
    GROUP BY v.user_id, v.first_view
),

-- User-level: first purchase after qualifying cart
user_purchase AS (
    SELECT
        c.user_id,
        MIN(b.event_time) AS first_purchase
    FROM user_cart c
    JOIN base b
      ON b.user_id = c.user_id
     AND b.event_type = 'purchase'
     AND b.event_time > c.first_cart
    GROUP BY c.user_id
),

-- Session-level: first view per user-session pair
session_view AS (
    SELECT
        user_id,
        user_session,
        MIN(event_time) AS first_view
    FROM base
    WHERE event_type = 'view'
      AND user_session IS NOT NULL
    GROUP BY user_id, user_session
),

-- Session-level: first cart after view in same session
session_cart AS (
    SELECT
        v.user_id,
        v.user_session,
        v.first_view,
        MIN(b.event_time) AS first_cart
    FROM session_view v
    JOIN base b
      ON b.user_id = v.user_id
     AND b.user_session = v.user_session
     AND b.event_type = 'cart'
     AND b.event_time > v.first_view
    GROUP BY v.user_id, v.user_session, v.first_view
),

-- Session-level: first purchase after cart in same session
session_purchase AS (
    SELECT
        c.user_id,
        c.user_session,
        MIN(b.event_time) AS first_purchase
    FROM session_cart c
    JOIN base b
      ON b.user_id = c.user_id
     AND b.user_session = c.user_session
     AND b.event_type = 'purchase'
     AND b.event_time > c.first_cart
    GROUP BY c.user_id, c.user_session
)

SELECT
    'User Funnel' AS funnel_type,
    (SELECT COUNT(*) FROM user_view) AS stage_1_view,
    (SELECT COUNT(*) FROM user_cart) AS stage_2_cart,
    (SELECT COUNT(*) FROM user_purchase) AS stage_3_purchase

UNION ALL

SELECT
    'Session Funnel',
    (SELECT COUNT(*) FROM session_view),
    (SELECT COUNT(*) FROM session_cart),
    (SELECT COUNT(*) FROM session_purchase)
"""

funnel_stage_independent = con.execute(
    query_funnel_stage_check,
    [OUTPUT_SESSION_FILE]
).df()

print("========== INDEPENDENT FUNNEL STAGE COUNTS ==========")
display(funnel_stage_independent)

# Compare with original funnel results
comparison = funnel_result[
    ["funnel_type", "stage_1_view", "stage_2_cart", "stage_3_purchase"]
].merge(
    funnel_stage_independent,
    on="funnel_type",
    suffixes=("_original", "_independent")
)

for stage in ["stage_1_view", "stage_2_cart", "stage_3_purchase"]:
    comparison[f"{stage}_difference"] = (
        comparison[f"{stage}_original"]
        - comparison[f"{stage}_independent"]
    )

print("========== ORIGINAL VS INDEPENDENT ==========")
display(comparison)

assert (
    comparison[
        [
            "stage_1_view_difference",
            "stage_2_cart_difference",
            "stage_3_purchase_difference"
        ]
    ] == 0
).all().all()

print("Independent stage reconciliation: PASS")


In [ ]:

# Phase 1.8.8.4 — Cross-Session Attribution Audit
# Corrected SQL aliases

query_cross_session_audit = """
WITH base AS (
    SELECT
        user_id,
        user_session,
        event_time,
        event_type
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
),

user_view AS (
    SELECT
        user_id,
        MIN(event_time) AS first_view
    FROM base
    WHERE event_type = 'view'
    GROUP BY user_id
),

user_cart AS (
    SELECT
        v.user_id,
        MIN(b.event_time) AS first_cart
    FROM user_view v
    JOIN base b
      ON b.user_id = v.user_id
     AND b.event_type = 'cart'
     AND b.event_time > v.first_view
    GROUP BY v.user_id
),

user_purchase AS (
    SELECT
        c.user_id,
        MIN(b.event_time) AS first_purchase
    FROM user_cart c
    JOIN base b
      ON b.user_id = c.user_id
     AND b.event_type = 'purchase'
     AND b.event_time > c.first_cart
    GROUP BY c.user_id
),

view_session AS (
    SELECT
        v.user_id,
        COUNT(DISTINCT b.user_session) AS session_count,
        COUNT(*) FILTER (
            WHERE b.user_session IS NULL
        ) AS null_session_events,
        MIN(b.user_session) AS single_session
    FROM user_view v
    JOIN base b
      ON b.user_id = v.user_id
     AND b.event_type = 'view'
     AND b.event_time = v.first_view
    GROUP BY v.user_id
),

cart_session AS (
    SELECT
        c.user_id,
        COUNT(DISTINCT b.user_session) AS session_count,
        COUNT(*) FILTER (
            WHERE b.user_session IS NULL
        ) AS null_session_events,
        MIN(b.user_session) AS single_session
    FROM user_cart c
    JOIN base b
      ON b.user_id = c.user_id
     AND b.event_type = 'cart'
     AND b.event_time = c.first_cart
    GROUP BY c.user_id
),

purchase_session AS (
    SELECT
        p.user_id,
        COUNT(DISTINCT b.user_session) AS session_count,
        COUNT(*) FILTER (
            WHERE b.user_session IS NULL
        ) AS null_session_events,
        MIN(b.user_session) AS single_session
    FROM user_purchase p
    JOIN base b
      ON b.user_id = p.user_id
     AND b.event_type = 'purchase'
     AND b.event_time = p.first_purchase
    GROUP BY p.user_id
)

SELECT
    COUNT(*) AS sequential_purchase_users,

    COUNT(*) FILTER (
        WHERE vs.session_count > 1
    ) AS ambiguous_view_timestamp_users,

    COUNT(*) FILTER (
        WHERE cs.session_count > 1
    ) AS ambiguous_cart_timestamp_users,

    COUNT(*) FILTER (
        WHERE ps.session_count > 1
    ) AS ambiguous_purchase_timestamp_users,

    COUNT(*) FILTER (
        WHERE vs.session_count = 1
          AND cs.session_count = 1
          AND vs.single_session <> cs.single_session
    ) AS view_to_cart_cross_session_users,

    COUNT(*) FILTER (
        WHERE cs.session_count = 1
          AND ps.session_count = 1
          AND cs.single_session <> ps.single_session
    ) AS cart_to_purchase_cross_session_users,

    COUNT(*) FILTER (
        WHERE vs.null_session_events > 0
    ) AS users_with_null_first_view_session,

    COUNT(*) FILTER (
        WHERE cs.null_session_events > 0
    ) AS users_with_null_first_cart_session,

    COUNT(*) FILTER (
        WHERE ps.null_session_events > 0
    ) AS users_with_null_first_purchase_session

FROM user_purchase p
JOIN user_view v ON p.user_id = v.user_id
JOIN user_cart c ON p.user_id = c.user_id
JOIN purchase_session ps ON p.user_id = ps.user_id
JOIN view_session vs ON p.user_id = vs.user_id
JOIN cart_session cs ON p.user_id = cs.user_id
"""

cross_session_audit = con.execute(
    query_cross_session_audit,
    [OUTPUT_SESSION_FILE]
).df()

print("========== CROSS-SESSION ATTRIBUTION AUDIT ==========")
display(cross_session_audit)


In [ ]:

# Phase 1.8.8.5 — Mutually Exclusive Cross-Session Classification

query_cross_session_classification = """
WITH base AS (
    SELECT user_id, user_session, event_time, event_type
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
),

user_view AS (
    SELECT user_id, MIN(event_time) AS first_view
    FROM base
    WHERE event_type = 'view'
    GROUP BY user_id
),

user_cart AS (
    SELECT
        v.user_id,
        MIN(b.event_time) AS first_cart
    FROM user_view v
    JOIN base b
      ON b.user_id = v.user_id
     AND b.event_type = 'cart'
     AND b.event_time > v.first_view
    GROUP BY v.user_id
),

user_purchase AS (
    SELECT
        c.user_id,
        MIN(b.event_time) AS first_purchase
    FROM user_cart c
    JOIN base b
      ON b.user_id = c.user_id
     AND b.event_type = 'purchase'
     AND b.event_time > c.first_cart
    GROUP BY c.user_id
),

view_session AS (
    SELECT
        v.user_id,
        COUNT(DISTINCT b.user_session) AS session_count,
        MIN(b.user_session) AS session_id
    FROM user_view v
    JOIN base b
      ON b.user_id = v.user_id
     AND b.event_type = 'view'
     AND b.event_time = v.first_view
    GROUP BY v.user_id
),

cart_session AS (
    SELECT
        c.user_id,
        COUNT(DISTINCT b.user_session) AS session_count,
        MIN(b.user_session) AS session_id
    FROM user_cart c
    JOIN base b
      ON b.user_id = c.user_id
     AND b.event_type = 'cart'
     AND b.event_time = c.first_cart
    GROUP BY c.user_id
),

purchase_session AS (
    SELECT
        p.user_id,
        COUNT(DISTINCT b.user_session) AS session_count,
        MIN(b.user_session) AS session_id
    FROM user_purchase p
    JOIN base b
      ON b.user_id = p.user_id
     AND b.event_type = 'purchase'
     AND b.event_time = p.first_purchase
    GROUP BY p.user_id
),

classified AS (
    SELECT
        p.user_id,

        CASE
            WHEN v.session_count > 1
              OR c.session_count > 1
              OR ps.session_count > 1
                THEN 'Ambiguous'

            WHEN v.session_id = c.session_id
             AND c.session_id = ps.session_id
                THEN 'Same Session'

            WHEN v.session_id <> c.session_id
             AND c.session_id = ps.session_id
                THEN 'View-Cart Cross Only'

            WHEN v.session_id = c.session_id
             AND c.session_id <> ps.session_id
                THEN 'Cart-Purchase Cross Only'

            WHEN v.session_id <> c.session_id
             AND c.session_id <> ps.session_id
                THEN 'Cross Both Transitions'

            ELSE 'Unclassified'
        END AS attribution_group

    FROM user_purchase p
    JOIN view_session v ON p.user_id = v.user_id
    JOIN cart_session c ON p.user_id = c.user_id
    JOIN purchase_session ps ON p.user_id = ps.user_id
)

SELECT
    attribution_group,
    COUNT(*) AS users,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        4
    ) AS pct_of_sequential_purchase_users
FROM classified
GROUP BY attribution_group
ORDER BY users DESC
"""

cross_session_classification = con.execute(
    query_cross_session_classification,
    [OUTPUT_SESSION_FILE]
).df()

print("========== CROSS-SESSION CLASSIFICATION ==========")
display(cross_session_classification)

print(
    "Total classified users:",
    cross_session_classification["users"].sum()
)

print(
    "Unclassified users:",
    int(
        cross_session_classification.loc[
            cross_session_classification["attribution_group"]
            == "Unclassified",
            "users"
        ].sum()
    )
)

assert cross_session_classification["users"].sum() == 196474
assert not (
    cross_session_classification["attribution_group"]
    == "Unclassified"
).any()

print("Mutually exclusive classification: PASS")


In [ ]:

# Phase 1.8.8.6 — Inspect 4 Ambiguous Users
# Corrected version

query_ambiguous_details = """
WITH base AS (
    SELECT user_id, user_session, event_time, event_type
    FROM read_parquet(?)
    WHERE user_id IS NOT NULL
),

user_view AS (
    SELECT user_id, MIN(event_time) AS first_view
    FROM base
    WHERE event_type = 'view'
    GROUP BY user_id
),

user_cart AS (
    SELECT
        v.user_id,
        MIN(b.event_time) AS first_cart
    FROM user_view v
    JOIN base b
      ON b.user_id = v.user_id
     AND b.event_type = 'cart'
     AND b.event_time > v.first_view
    GROUP BY v.user_id
),

user_purchase AS (
    SELECT
        c.user_id,
        MIN(b.event_time) AS first_purchase
    FROM user_cart c
    JOIN base b
      ON b.user_id = c.user_id
     AND b.event_type = 'purchase'
     AND b.event_time > c.first_cart
    GROUP BY c.user_id
),

ambiguous_users AS (
    SELECT
        v.user_id,
        v.first_view
    FROM user_view v
    JOIN user_cart c
      ON v.user_id = c.user_id
    JOIN user_purchase p
      ON v.user_id = p.user_id
    JOIN base b
      ON b.user_id = v.user_id
     AND b.event_type = 'view'
     AND b.event_time = v.first_view
    GROUP BY v.user_id, v.first_view
    HAVING COUNT(DISTINCT b.user_session) > 1
)

SELECT
    a.user_id,
    a.first_view,
    c.first_cart,
    p.first_purchase,

    COUNT(DISTINCT b.user_session)
        AS first_view_session_count,

    LIST(DISTINCT b.user_session ORDER BY b.user_session)
        AS first_view_sessions

FROM ambiguous_users a

JOIN user_cart c
  ON a.user_id = c.user_id

JOIN user_purchase p
  ON a.user_id = p.user_id

JOIN base b
  ON b.user_id = a.user_id
 AND b.event_type = 'view'
 AND b.event_time = a.first_view

GROUP BY
    a.user_id,
    a.first_view,
    c.first_cart,
    p.first_purchase

ORDER BY a.user_id
"""

ambiguous_details = con.execute(
    query_ambiguous_details,
    [OUTPUT_SESSION_FILE]
).df()

print("========== AMBIGUOUS USER DETAILS ==========")
display(ambiguous_details)

print("\nAmbiguous users:", len(ambiguous_details))

assert len(ambiguous_details) == 4

print("Ambiguous user count validation: PASS")


In [ ]:

# Phase 1.8.8.7 — Funnel Final Output Export

# Table A: Overall Funnel
funnel_overall_export = funnel_result[
    [
        "funnel_type",
        "stage_1_view",
        "stage_2_cart",
        "stage_3_purchase",
        "view_to_cart_rate",
        "cart_to_purchase_rate",
        "view_to_purchase_rate"
    ]
].copy()

# Table B: Cross-session attribution
funnel_attribution_export = cross_session_classification.copy()

# Add observation boundary
funnel_overall_export["max_observation_date"] = (
    pd.Timestamp("2019-10-31")
)

# Export
funnel_overall_export.to_csv(
    "Funnel_Overall.csv",
    index=False
)

funnel_attribution_export.to_csv(
    "Funnel_Attribution.csv",
    index=False
)

# Basic integrity checks
assert len(funnel_overall_export) == 2
assert len(funnel_attribution_export) == 5

assert (
    funnel_overall_export["stage_1_view"]
    >= funnel_overall_export["stage_2_cart"]
).all()

assert (
    funnel_overall_export["stage_2_cart"]
    >= funnel_overall_export["stage_3_purchase"]
).all()

assert funnel_attribution_export["users"].sum() == 196474

assert not (
    funnel_attribution_export["attribution_group"]
    == "Unclassified"
).any()

print("========== FUNNEL EXPORT VALIDATION ==========")
print("Overall Funnel rows:", len(funnel_overall_export))
print("Attribution rows:", len(funnel_attribution_export))
print("Sequential stage validation: PASS")
print("Attribution reconciliation: PASS")

print("\n========== OVERALL FUNNEL ==========")
display(funnel_overall_export)

print("\n========== ATTRIBUTION ==========")
display(funnel_attribution_export)

print("\nExport completed.")


In [ ]:

# Phase 1.8.8.8 — Final Funnel CSV Integrity Audit

from pathlib import Path
import pandas as pd

overall_path = Path("Funnel_Overall.csv")
attribution_path = Path("Funnel_Attribution.csv")

assert overall_path.exists()
assert attribution_path.exists()

overall = pd.read_csv(overall_path)
attribution = pd.read_csv(attribution_path)

# 1. Schema
expected_overall = {
    "funnel_type",
    "stage_1_view",
    "stage_2_cart",
    "stage_3_purchase",
    "view_to_cart_rate",
    "cart_to_purchase_rate",
    "view_to_purchase_rate",
    "max_observation_date"
}

expected_attribution = {
    "attribution_group",
    "users",
    "pct_of_sequential_purchase_users"
}

assert expected_overall.issubset(overall.columns)
assert expected_attribution.issubset(attribution.columns)

# 2. Row counts and unique categories
assert len(overall) == 2
assert len(attribution) == 5
assert overall["funnel_type"].is_unique
assert attribution["attribution_group"].is_unique

# 3. Funnel population
assert overall.loc[
    overall["funnel_type"] == "User Funnel",
    "stage_3_purchase"
].iloc[0] == 196474

assert overall.loc[
    overall["funnel_type"] == "Session Funnel",
    "stage_3_purchase"
].iloc[0] == 288767

# 4. Sequential stage validation
assert (
    overall["stage_1_view"] >= overall["stage_2_cart"]
).all()

assert (
    overall["stage_2_cart"] >= overall["stage_3_purchase"]
).all()

# 5. Attribution reconciliation
assert attribution["users"].sum() == 196474

assert set(attribution["attribution_group"]) == {
    "View-Cart Cross Only",
    "Same Session",
    "Cross Both Transitions",
    "Cart-Purchase Cross Only",
    "Ambiguous"
}

assert (
    attribution["pct_of_sequential_purchase_users"].sum()
    .round(4) == 100.0000
)

# 6. Observation boundary
assert (
    overall["max_observation_date"].astype(str)
    == "2019-10-31"
).all()

# 7. Final report
print("========== FINAL FUNNEL CSV AUDIT ==========")
print("Schema validation: PASS")
print("Row count validation: PASS")
print("Unique category validation: PASS")
print("Funnel population validation: PASS")
print("Sequential stage validation: PASS")
print("Attribution reconciliation: PASS")
print("Observation boundary validation: PASS")

print("\n========== FILE INFORMATION ==========")
print("Overall:", overall_path.resolve())
print("Attribution:", attribution_path.resolve())
print("Overall rows:", len(overall))
print("Attribution rows:", len(attribution))

print("\nFINAL FUNNEL CSV AUDIT: PASS")


In [ ]:

# ============================================================
# FIX — Power BI User Analysis Export
# Full user data -> CSV
# Summary tables -> Excel
# ============================================================

import os
import pandas as pd

EXPORT_DIR = r"D:\PowerBI_Project\PowerBI_Export\User_Analysis"
os.makedirs(EXPORT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Export full User Dimension to CSV
# ------------------------------------------------------------

user_export.to_csv(
    os.path.join(EXPORT_DIR, "User_Dimension.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("User_Dimension.csv exported")
print("Rows:", len(user_export))


# ------------------------------------------------------------
# 2. Export Top Users to CSV
# ------------------------------------------------------------

top_users.to_csv(
    os.path.join(EXPORT_DIR, "User_Top_Users.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. Export all summary CSV files
# ------------------------------------------------------------

csv_tables = {
    "User_Purchase_Summary.csv": purchase_summary,
    "User_Activity_Summary.csv": activity_summary,
    "User_Funnel.csv": user_funnel,
    "User_Retention_Cohort.csv": user_retention,
    "User_Retention_Overall.csv": overall_retention,
    "User_Reconciliation.csv": user_reconciliation
}

for filename, df in csv_tables.items():
    df.to_csv(
        os.path.join(EXPORT_DIR, filename),
        index=False,
        encoding="utf-8-sig"
    )
    print("Exported:", filename)


# ------------------------------------------------------------
# 4. Export Excel — summary tables only
# ------------------------------------------------------------

excel_path = os.path.join(
    EXPORT_DIR,
    "PowerBI_User_Analysis.xlsx"
)

excel_tables = {
    "Purchase_Summary": purchase_summary,
    "Activity_Summary": activity_summary,
    "Top_Users": top_users,
    "User_Funnel": user_funnel,
    "Retention_Cohort": user_retention,
    "Retention_Overall": overall_retention,
    "Reconciliation": user_reconciliation
}

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    for sheet_name, df in excel_tables.items():
        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

print("\nExcel workbook exported:", excel_path)


# ------------------------------------------------------------
# 5. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("EXPORT VALIDATION")
print("=" * 55)

print("User Dimension rows:", len(user_export))
print("Unique users:", user_export["user_id"].nunique())
print("Duplicate user IDs:", user_export["user_id"].duplicated().sum())

print("\nReconciliation:")
display(user_reconciliation)

print("\nFiles created:")
for filename in sorted(os.listdir(EXPORT_DIR)):
    filepath = os.path.join(EXPORT_DIR, filename)
    print(f"{filename} | {os.path.getsize(filepath) / 1024 / 1024:.2f} MB")

print("\nExport completed.")


In [ ]:

# ============================================================
# Phase 1 — RFM Analysis & User Value Segmentation
# Dataset: ecommerce_session_ready.parquet
# Output: PowerBI_Export/User_Analysis/RFM
# ============================================================

import os
import numpy as np
import pandas as pd
import duckdb

# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

RFM_DIR = os.path.join(EXPORT_DIR, "RFM")
os.makedirs(RFM_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. Extract actual purchase-level RFM metrics
# ------------------------------------------------------------

query_rfm = """
SELECT
    user_id,

    COUNT(*) AS frequency,

    SUM(price) AS monetary,

    MIN(event_time) AS first_purchase_date,

    MAX(event_time) AS last_purchase_date

FROM read_parquet(?)

WHERE event_type = 'purchase'
  AND user_id IS NOT NULL

GROUP BY user_id
"""

rfm_raw = con.execute(
    query_rfm,
    [OUTPUT_SESSION_FILE]
).df()

rfm_raw["first_purchase_date"] = pd.to_datetime(
    rfm_raw["first_purchase_date"]
)

rfm_raw["last_purchase_date"] = pd.to_datetime(
    rfm_raw["last_purchase_date"]
)

print("Purchasing users:", len(rfm_raw))
print("Unique purchasing users:", rfm_raw["user_id"].nunique())


# ------------------------------------------------------------
# 3. Define observation reference date
# Use maximum observed event date + 1 day
# ------------------------------------------------------------

max_event_date = pd.to_datetime(
    con.execute(
        """
        SELECT MAX(event_time)
        FROM read_parquet(?)
        """,
        [OUTPUT_SESSION_FILE]
    ).fetchone()[0]
)

reference_date = max_event_date.normalize() + pd.Timedelta(days=1)

print("Maximum event date:", max_event_date)
print("RFM reference date:", reference_date)


# ------------------------------------------------------------
# 4. Calculate Recency
# ------------------------------------------------------------

rfm_raw["recency"] = (
    reference_date.normalize()
    - rfm_raw["last_purchase_date"].dt.normalize()
).dt.days

# Data quality checks
assert (rfm_raw["recency"] >= 0).all()
assert (rfm_raw["frequency"] > 0).all()
assert (rfm_raw["monetary"] >= 0).all()


# ------------------------------------------------------------
# 5. Calculate R, F, M scores (1–5)
# Percentile ranking handles tied values more robustly
# ------------------------------------------------------------

# Recency: fewer days since purchase = higher score
rfm_raw["R_Score"] = np.ceil(
    rfm_raw["recency"].rank(
        ascending=False,
        method="average",
        pct=True
    ) * 5
).clip(1, 5).astype(int)

# Frequency: more purchases = higher score
rfm_raw["F_Score"] = np.ceil(
    rfm_raw["frequency"].rank(
        ascending=True,
        method="average",
        pct=True
    ) * 5
).clip(1, 5).astype(int)

# Monetary: higher spending = higher score
rfm_raw["M_Score"] = np.ceil(
    rfm_raw["monetary"].rank(
        ascending=True,
        method="average",
        pct=True
    ) * 5
).clip(1, 5).astype(int)


# ------------------------------------------------------------
# 6. Calculate RFM total score
# ------------------------------------------------------------

rfm_raw["RFM_Score"] = (
    rfm_raw["R_Score"]
    + rfm_raw["F_Score"]
    + rfm_raw["M_Score"]
)


# ------------------------------------------------------------
# 7. Assign value segments
# ------------------------------------------------------------

rfm_raw["RFM_Segment"] = np.select(
    [
        rfm_raw["RFM_Score"] <= 8,
        rfm_raw["RFM_Score"].between(9, 12),
        rfm_raw["RFM_Score"] >= 13
    ],
    [
        "Low Value",
        "Medium Value",
        "High Value"
    ],
    default="Unclassified"
)


# ------------------------------------------------------------
# 8. Merge with all users
# Non-purchasing users remain a separate segment
# ------------------------------------------------------------

user_segmentation = user_export.merge(
    rfm_raw[
        [
            "user_id",
            "recency",
            "frequency",
            "monetary",
            "first_purchase_date",
            "last_purchase_date",
            "R_Score",
            "F_Score",
            "M_Score",
            "RFM_Score",
            "RFM_Segment"
        ]
    ],
    on="user_id",
    how="left",
    validate="one_to_one"
)

user_segmentation["RFM_Segment"] = (
    user_segmentation["RFM_Segment"]
    .fillna("No Purchase")
)

# ------------------------------------------------------------
# 9. Segment summary
# ------------------------------------------------------------

segment_order = [
    "No Purchase",
    "Low Value",
    "Medium Value",
    "High Value"
]

segment_summary = (
    user_segmentation.groupby("RFM_Segment")
    .agg(
        users=("user_id", "nunique"),
        purchase_events=("purchase_events", "sum"),
        total_spent=("total_spent", "sum"),
        avg_spend_per_user=("total_spent", "mean"),
        median_spend_per_user=("total_spent", "median"),
        avg_purchase_frequency=("purchase_events", "mean"),
        avg_recency_days=("recency", "mean")
    )
    .reindex(segment_order)
    .reset_index()
)

segment_summary["user_percentage"] = (
    segment_summary["users"]
    / segment_summary["users"].sum() * 100
).round(4)

total_gmv = segment_summary["total_spent"].sum()

segment_summary["gmv_percentage"] = (
    segment_summary["total_spent"]
    / total_gmv * 100
).round(4)


# ------------------------------------------------------------
# 10. RFM score distribution
# ------------------------------------------------------------

rfm_score_distribution = (
    rfm_raw.groupby("RFM_Score")
    .agg(
        users=("user_id", "nunique"),
        purchase_events=("frequency", "sum"),
        monetary=("monetary", "sum")
    )
    .reset_index()
)

rfm_score_distribution["user_percentage"] = (
    rfm_score_distribution["users"]
    / rfm_score_distribution["users"].sum() * 100
).round(4)


# ------------------------------------------------------------
# 11. RFM score profile by segment
# ------------------------------------------------------------

rfm_profile = (
    rfm_raw.groupby("RFM_Segment")
    .agg(
        users=("user_id", "nunique"),
        avg_R=("R_Score", "mean"),
        avg_F=("F_Score", "mean"),
        avg_M=("M_Score", "mean"),
        avg_RFM_Score=("RFM_Score", "mean"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 12. Export CSV files
# ------------------------------------------------------------

user_segmentation.to_csv(
    os.path.join(RFM_DIR, "User_RFM_Segmentation.csv"),
    index=False,
    encoding="utf-8-sig"
)

rfm_raw.to_csv(
    os.path.join(RFM_DIR, "RFM_Purchasing_Users.csv"),
    index=False,
    encoding="utf-8-sig"
)

segment_summary.to_csv(
    os.path.join(RFM_DIR, "RFM_Segment_Summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

rfm_score_distribution.to_csv(
    os.path.join(RFM_DIR, "RFM_Score_Distribution.csv"),
    index=False,
    encoding="utf-8-sig"
)

rfm_profile.to_csv(
    os.path.join(RFM_DIR, "RFM_Segment_Profile.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 13. Export summary workbook
# ------------------------------------------------------------

rfm_excel_path = os.path.join(
    RFM_DIR,
    "RFM_Analysis_Summary.xlsx"
)

with pd.ExcelWriter(
    rfm_excel_path,
    engine="openpyxl"
) as writer:

    segment_summary.to_excel(
        writer,
        sheet_name="Segment_Summary",
        index=False
    )

    rfm_score_distribution.to_excel(
        writer,
        sheet_name="Score_Distribution",
        index=False
    )

    rfm_profile.to_excel(
        writer,
        sheet_name="Segment_Profile",
        index=False
    )


# ------------------------------------------------------------
# 14. Validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RFM SEGMENTATION VALIDATION")
print("=" * 65)

print("Total users:", len(user_segmentation))
print("Unique users:", user_segmentation["user_id"].nunique())
print("Duplicate users:", user_segmentation["user_id"].duplicated().sum())

print("\nPurchase users:", len(rfm_raw))
print("Non-purchase users:", (user_segmentation["RFM_Segment"] == "No Purchase").sum())

print("\nSegment Summary:")
display(segment_summary)

print("\nRFM Score Distribution:")
display(rfm_score_distribution)

print("\nRFM Profile:")
display(rfm_profile)

print("\nRFM score boundaries:")
print("Minimum:", rfm_raw["RFM_Score"].min())
print("Maximum:", rfm_raw["RFM_Score"].max())

print("\nRFM reference date:", reference_date)

print("\nExported files:")
for filename in sorted(os.listdir(RFM_DIR)):
    path = os.path.join(RFM_DIR, filename)
    print(
        filename,
        "|",
        round(os.path.getsize(path) / 1024 / 1024, 2),
        "MB"
    )

print("\nRFM analysis completed.")


In [ ]:

# ============================================================
# Page 1 — Executive Overview
# Memory-Optimized Export
# ============================================================

from pathlib import Path
import pandas as pd
import gc

OVERVIEW_DIR = Path(
    r"D:\PowerBI_Project\PowerBI_Export\Executive_Overview"
)
OVERVIEW_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILE = OUTPUT_SESSION_FILE

# Reduce DuckDB memory pressure
con.execute("SET memory_limit = '1GB'")
con.execute("SET threads = 2")

# ------------------------------------------------------------
# 1. Overview KPI
# Reuse the already validated Cell 53 result
# ------------------------------------------------------------

overview_kpi = global_user_check.copy()

overview_kpi.to_csv(
    OVERVIEW_DIR / "Overview_KPI.csv",
    index=False,
    encoding="utf-8-sig"
)

print("========== OVERVIEW KPI ==========")
display(overview_kpi)


# ------------------------------------------------------------
# 2. Event Summary
# Low-cardinality aggregation only
# ------------------------------------------------------------

query_event_summary = """
SELECT
    event_type,
    COUNT(*) AS event_count,

    SUM(
        CASE
            WHEN event_type = 'purchase'
            THEN price
            ELSE 0
        END
    ) AS event_gmv

FROM read_parquet(?)

GROUP BY event_type
ORDER BY event_count DESC
"""

overview_event_summary = con.execute(
    query_event_summary,
    [SOURCE_FILE]
).df()

overview_event_summary["event_gmv"] = (
    overview_event_summary["event_gmv"].round(2)
)

overview_event_summary.to_csv(
    OVERVIEW_DIR / "Overview_Event_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n========== EVENT SUMMARY ==========")
display(overview_event_summary)


# ------------------------------------------------------------
# 3. Daily Trend
# No daily COUNT DISTINCT
# ------------------------------------------------------------

query_daily_trend = """
SELECT
    CAST(event_time AS DATE) AS event_date,

    COUNT(*) AS total_events,

    COUNT(*) FILTER (
        WHERE event_type = 'view'
    ) AS view_events,

    COUNT(*) FILTER (
        WHERE event_type = 'cart'
    ) AS cart_events,

    COUNT(*) FILTER (
        WHERE event_type = 'purchase'
    ) AS purchase_events,

    SUM(
        CASE
            WHEN event_type = 'purchase'
            THEN price
            ELSE 0
        END
    ) AS daily_gmv

FROM read_parquet(?)

GROUP BY CAST(event_time AS DATE)

ORDER BY event_date
"""

overview_daily_trend = con.execute(
    query_daily_trend,
    [SOURCE_FILE]
).df()

overview_daily_trend["daily_gmv"] = (
    overview_daily_trend["daily_gmv"].round(2)
)

overview_daily_trend.to_csv(
    OVERVIEW_DIR / "Overview_Daily_Trend.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n========== DAILY TREND ==========")
print("Rows:", len(overview_daily_trend))
print(
    "Date range:",
    overview_daily_trend["event_date"].min(),
    "to",
    overview_daily_trend["event_date"].max()
)
display(overview_daily_trend.head(10))


# ------------------------------------------------------------
# 4. Reconciliation
# ------------------------------------------------------------

print("\n========== VALIDATION ==========")

kpi = overview_kpi.iloc[0]

print(
    "Total Events:",
    int(kpi["total_events"])
)

print(
    "Unique Users:",
    int(kpi["unique_users"])
)

print(
    "Purchase Events:",
    int(kpi["purchase_events"])
)

print(
    "Total GMV:",
    round(float(kpi["purchase_event_gmv"]), 2)
)

daily_event_check = (
    int(overview_daily_trend["total_events"].sum())
    == int(kpi["total_events"])
)

daily_gmv_check = (
    abs(
        overview_daily_trend["daily_gmv"].sum()
        - float(kpi["purchase_event_gmv"])
    ) < 0.01
)

print("Daily event reconciliation:", daily_event_check)
print("Daily GMV reconciliation:", daily_gmv_check)

print("\n========== EXPORTED FILES ==========")

for file in OVERVIEW_DIR.glob("*.csv"):
    print(
        file.name,
        "|",
        round(file.stat().st_size / 1024, 2),
        "KB"
    )

gc.collect()


In [8]:

# ============================================================
# Cross-Page Consistency Audit
# CSV-based validation — no large Parquet aggregation
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path(r"D:\PowerBI_Project\PowerBI_Export")
USER_DIR = BASE_DIR / "User_Analysis"
RFM_DIR = USER_DIR / "RFM"
OVERVIEW_DIR = BASE_DIR / "Executive_Overview"

# ------------------------------------------------------------
# 1. Load exported summary files
# ------------------------------------------------------------

overview = pd.read_csv(
    OVERVIEW_DIR / "Overview_KPI.csv"
)

user_reconciliation = pd.read_csv(
    USER_DIR / "User_Reconciliation.csv"
)

rfm_summary = pd.read_csv(
    RFM_DIR / "RFM_Segment_Summary.csv"
)

print("Files loaded successfully.")


# ------------------------------------------------------------
# 2. Page 1 KPI values
# ------------------------------------------------------------

p1 = overview.iloc[0]

# ------------------------------------------------------------
# 3. Page 3 User Reconciliation
# ------------------------------------------------------------

print("\n========== USER RECONCILIATION FILE ==========")
display(user_reconciliation)

# Automatically identify columns from the exported audit
print("Columns:", user_reconciliation.columns.tolist())


# ------------------------------------------------------------
# 4. RFM Reconciliation
# ------------------------------------------------------------

rfm_all_users = rfm_summary["users"].sum()

rfm_purchase = rfm_summary[
    rfm_summary["RFM_Segment"].isin(
        ["Low Value", "Medium Value", "High Value"]
    )
]

rfm_purchase_users = rfm_purchase["users"].sum()
rfm_purchase_events = rfm_purchase["purchase_events"].sum()
rfm_gmv = rfm_purchase["total_spent"].sum()

rfm_audit = pd.DataFrame([
    {
        "metric": "RFM Total Users",
        "value": rfm_all_users,
        "expected": p1["unique_users"]
    },
    {
        "metric": "RFM Purchase Users",
        "value": rfm_purchase_users,
        "expected": 347118
    },
    {
        "metric": "RFM Purchase Events",
        "value": rfm_purchase_events,
        "expected": p1["purchase_events"]
    },
    {
        "metric": "RFM GMV",
        "value": rfm_gmv,
        "expected": p1["purchase_event_gmv"]
    }
])

rfm_audit["difference"] = (
    rfm_audit["value"] - rfm_audit["expected"]
)

rfm_audit["status"] = np.where(
    rfm_audit["difference"].abs() < 0.01,
    "PASS",
    "CHECK"
)

print("\n========== RFM RECONCILIATION ==========")
display(rfm_audit)


# ------------------------------------------------------------
# 5. Page 1 daily trend reconciliation
# ------------------------------------------------------------

daily = pd.read_csv(
    OVERVIEW_DIR / "Overview_Daily_Trend.csv"
)

daily_audit = pd.DataFrame([
    {
        "metric": "Total Events",
        "daily_sum": daily["total_events"].sum(),
        "overview_value": p1["total_events"]
    },
    {
        "metric": "View Events",
        "daily_sum": daily["view_events"].sum(),
        "overview_value": p1["view_events"]
    },
    {
        "metric": "Cart Events",
        "daily_sum": daily["cart_events"].sum(),
        "overview_value": p1["cart_events"]
    },
    {
        "metric": "Purchase Events",
        "daily_sum": daily["purchase_events"].sum(),
        "overview_value": p1["purchase_events"]
    },
    {
        "metric": "GMV",
        "daily_sum": daily["daily_gmv"].sum(),
        "overview_value": p1["purchase_event_gmv"]
    }
])

daily_audit["difference"] = (
    daily_audit["daily_sum"] - daily_audit["overview_value"]
)

daily_audit["status"] = np.where(
    daily_audit["difference"].abs() < 0.01,
    "PASS",
    "CHECK"
)

print("\n========== DAILY TREND AUDIT ==========")
display(daily_audit)


# ------------------------------------------------------------
# 6. Save audit outputs
# ------------------------------------------------------------

rfm_audit.to_csv(
    OVERVIEW_DIR / "Audit_RFM_Reconciliation.csv",
    index=False,
    encoding="utf-8-sig"
)

daily_audit.to_csv(
    OVERVIEW_DIR / "Audit_Daily_Reconciliation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAudit files exported.")


Files loaded successfully.

========== USER RECONCILIATION FILE ==========


,metric,user_dimension,global_events,difference
0,Unique Users,3.022290e+06,3.022290e+06,0.000000
1,Total Events,4.241854e+07,4.241854e+07,0.000000
2,View Events,4.077733e+07,4.077733e+07,0.000000
3,Cart Events,8.984430e+05,8.984430e+05,0.000000
4,Purchase Events,7.427730e+05,7.427730e+05,0.000000
5,User-Session Pairs,9.244770e+06,9.244770e+06,0.000000
6,Purchase-event GMV,2.299332e+08,2.299332e+08,-0.000012


Columns: ['metric', 'user_dimension', 'global_events', 'difference']

========== RFM RECONCILIATION ==========


,metric,value,expected,difference,status
0,RFM Total Users,3.022290e+06,3.022290e+06,0.000000,PASS
1,RFM Purchase Users,3.471180e+05,3.471180e+05,0.000000,PASS
2,RFM Purchase Events,7.427730e+05,7.427730e+05,0.000000,PASS
3,RFM GMV,2.299332e+08,2.299332e+08,-0.000012,PASS



========== DAILY TREND AUDIT ==========


,metric,daily_sum,overview_value,difference,status
0,Total Events,4.241854e+07,4.241854e+07,0.000000,PASS
1,View Events,4.077733e+07,4.077733e+07,0.000000,PASS
2,Cart Events,8.984430e+05,8.984430e+05,0.000000,PASS
3,Purchase Events,7.427730e+05,7.427730e+05,0.000000,PASS
4,GMV,2.299332e+08,2.299332e+08,-0.000012,PASS



Audit files exported.
